# Huấn luyện Mô hình CRAM-ABSA (v2)


In [ ]:
import os

os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ.setdefault("PYTORCH_MPS_HIGH_WATERMARK_RATIO", "0.7")
os.environ.setdefault("PYTORCH_MPS_LOW_WATERMARK_RATIO", "0.5")

import gc
import re
import sys
import math
import json
import time
import shutil
import pickle
import random
import platform
import unicodedata
from pathlib import Path
from contextlib import nullcontext
from collections import Counter, defaultdict
from typing import List, Dict, Tuple, Optional, Any

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_cosine_schedule_with_warmup
from transformers import logging as hf_logging
from sklearn.metrics import f1_score, mean_absolute_error, accuracy_score, cohen_kappa_score

hf_logging.set_verbosity_error()

import nltk
for pkg in ["stopwords", "punkt", "averaged_perceptron_tagger", "wordnet"]:
    try:
        nltk.data.find(f"corpora/{pkg}" if pkg in ["stopwords", "wordnet"] else f"tokenizers/{pkg}")
    except LookupError:
        try:
            nltk.download(pkg, quiet=True)
        except Exception:
            pass

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

MPS_BUILT = hasattr(torch.backends, "mps") and torch.backends.mps.is_built()
MPS_AVAILABLE = hasattr(torch.backends, "mps") and torch.backends.mps.is_available()

if MPS_AVAILABLE:
    DEVICE = torch.device("mps")
    if hasattr(torch, "mps") and hasattr(torch.mps, "manual_seed"):
        torch.mps.manual_seed(SEED)
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
else:
    DEVICE = torch.device("cpu")

NUM_GPUS = torch.cuda.device_count() if DEVICE.type == "cuda" else 0
print(
    f"PyTorch: {torch.__version__} | Arch: {platform.machine()} | "
    f"Device: {DEVICE} | MPS built={MPS_BUILT}, available={MPS_AVAILABLE} | CUDA GPUs: {NUM_GPUS}"
)


## 1. Thiết lập Cấu hình Huấn luyện CRAM-ABSA v2 (MacBook Air M3 8GB / MPS)


In [ ]:
def resolve_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, cwd.parent, cwd.parent.parent):
        if (candidate / "data" / "df" / "hotel_reviews.parquet").exists():
            return candidate
    for candidate in (cwd, cwd.parent, cwd.parent.parent):
        if (candidate / "train" / "train_model").exists() and (candidate / "data").exists():
            return candidate
    return cwd.parent.parent if cwd.name == "train_model" else cwd


REPO_ROOT = resolve_repo_root()


class TrainConfig:
    REPO_ROOT = REPO_ROOT
    PARQUET_CANDIDATES = [
        REPO_ROOT / "data/df/hotel_reviews.parquet",
        Path("../../data/df/hotel_reviews.parquet"),
        Path("data/df/hotel_reviews.parquet"),
        REPO_ROOT / "hotel_reviews.parquet",
        Path("hotel_reviews.parquet"),
    ]
    OUTPUT_DIR = REPO_ROOT / "model" / "modelv2"

    MODEL_FILENAME = "modelv2.pt"
    MODEL_SAVE_PATH = OUTPUT_DIR / "modelv2.pt"
    LAST_CKPT_PATH = OUTPUT_DIR / "last_checkpoint_v2.pt"
    EBR_BANK_SAVE_PATH = OUTPUT_DIR / "ebr_exemplar_bank_v2.pkl"
    HISTORY_SAVE_PATH = OUTPUT_DIR / "training_history_v2.json"
    ANE_SAVE_PATH = OUTPUT_DIR / "modelv2_ane.mlpackage"

    BACKBONE_NAME = "bert-base-uncased"
    HIDDEN_DIM = 768
    NUM_FEATURES = 20
    PROJ_DIM = 256
    ATTN_HEADS = 8
    CANONICAL_ASPECTS = ["Service", "Facility", "Experience", "Amenity", "Loyalty", "Branding"]
    NUM_ASPECTS = 6
    NUM_SENTIMENT_CLASSES = 4
    NUM_POLARITY_CLASSES = 3
    MAX_SEQ_LEN = 192
    MAX_COMMENT_LEN = 48

    PER_GPU_BATCH_SIZE = 4 if DEVICE.type == "mps" else 16
    BATCH_SIZE = PER_GPU_BATCH_SIZE * max(1, NUM_GPUS)
    EVAL_BATCH_SIZE = 8 if DEVICE.type == "mps" else BATCH_SIZE
    GRAD_ACCUM_STEPS = 8 if DEVICE.type == "mps" else 2
    NUM_WORKERS = 0
    PIN_MEMORY = (DEVICE.type == "cuda")
    USE_AMP = (DEVICE.type in ("cuda", "mps"))
    AMP_DTYPE = torch.bfloat16 if DEVICE.type == "mps" else torch.float16
    USE_GRAD_CHECKPOINTING = (DEVICE.type == "mps")
    USE_DYNAMIC_PADDING = True
    MPS_EMPTY_CACHE_EVERY = 50
    RESUME_FROM_LAST = True
    EARLY_STOPPING_PATIENCE = 3
    PROGRESS_BAR_WIDTH = 28

    EPOCHS = 8
    LR_BACKBONE = 3e-5
    LR_HEADS = 1e-4
    WEIGHT_DECAY = 0.01
    WARMUP_RATIO = 0.10
    LABEL_SMOOTHING = 0.03
    MAX_GRAD_NORM = 1.0

    BETA_CORN = 0.20
    BETA_OVERLAP = 0.35
    BETA_RATIONALE = 0.25
    BETA_COMMENTARY = 0.05
    BETA_MULTI_POL = 0.15

    FEATURE_COLS_20 = [
        "feat_aspect_count",
        "feat_dominant_aspect",
        "feat_net_sentiment_score",
        "feat_sentiment_conflict",
        "feat_evidence_density",
        "feat_avg_rationale_len",
        "feat_has_cross_overlap",
        "feat_service_sentiment",
        "feat_facility_sentiment",
        "feat_experience_sentiment",
        "feat_amenity_sentiment",
        "feat_loyalty_sentiment",
        "feat_branding_sentiment",
        "feat_aspect_coverage_ratio",
        "feat_neutral_hedging_density",
        "feat_consensus_strength",
        "feat_lexical_polarity_intensity",
        "feat_nltk_clean_token_ratio",
        "feat_nltk_opinion_negation_density",
        "feat_ebr_aspect_prototype_sim",
    ]


cfg = TrainConfig()
cfg.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(
    f"REPO_ROOT: {cfg.REPO_ROOT} | OUTPUT_DIR: {cfg.OUTPUT_DIR}\n"
    f"Model save path: {cfg.MODEL_SAVE_PATH} | Last ckpt path: {cfg.LAST_CKPT_PATH} | ANE export path: {cfg.ANE_SAVE_PATH}\n"
    f"MAX_SEQ_LEN={cfg.MAX_SEQ_LEN} | Batch size: {cfg.BATCH_SIZE} x Accum {cfg.GRAD_ACCUM_STEPS} "
    f"(eff={cfg.BATCH_SIZE * cfg.GRAD_ACCUM_STEPS}) | EVAL_BATCH_SIZE={cfg.EVAL_BATCH_SIZE}\n"
    f"PIN_MEMORY={cfg.PIN_MEMORY} | USE_AMP={cfg.USE_AMP} ({cfg.AMP_DTYPE}) | "
    f"USE_GRAD_CHECKPOINTING={cfg.USE_GRAD_CHECKPOINTING} | USE_DYNAMIC_PADDING={cfg.USE_DYNAMIC_PADDING}"
)


## 2. Xử lý Ngôn ngữ Tự nhiên Dual-Stream NLTK, Clause Splitter Hybrid & Trích xuất 20 Đặc trưng


In [ ]:
try:
    from nltk.corpus import stopwords
    from nltk.stem import WordNetLemmatizer
    RAW_NLTK_STOPWORDS = set(stopwords.words("english"))
    NLTK_LEMMATIZER = WordNetLemmatizer()
    _ = NLTK_LEMMATIZER.lemmatize("rooms")
    def lemmatize_word(w: str) -> str:
        try:
            return NLTK_LEMMATIZER.lemmatize(w)
        except Exception:
            return w[:-1] if (len(w) > 3 and w.endswith("s") and not w.endswith("ss")) else w
except Exception:
    RAW_NLTK_STOPWORDS = {
        "a", "an", "the", "and", "or", "if", "in", "on", "at", "to", "from", "by", "for",
        "with", "about", "as", "into", "like", "through", "after", "over", "between", "out",
        "of", "during", "before", "under", "around", "among", "is", "are", "was", "were",
        "be", "been", "being", "have", "has", "had", "do", "does", "did", "will", "would",
        "shall", "should", "can", "could", "may", "might", "must", "i", "me", "my", "myself",
        "we", "our", "ours", "ourselves", "you", "your", "yours", "he", "him", "his", "she",
        "her", "hers", "it", "its", "they", "them", "their", "theirs", "this", "that", "these",
        "those", "am", "so", "than", "too", "very", "just", "where", "when", "who", "whom",
        "which", "what", "there", "here", "all", "any", "both", "each", "few", "more", "most",
        "other", "some", "such", "only", "own", "same", "then", "once"
    }
    def lemmatize_word(w: str) -> str:
        if len(w) > 4 and w.endswith("ies"):
            return w[:-3] + "y"
        if len(w) > 3 and w.endswith("s") and not w.endswith("ss"):
            return w[:-1]
        return w

CONTRACTIONS_MAP = {
    "can't": "cannot", "won't": "will not", "don't": "do not", "doesn't": "does not",
    "didn't": "did not", "isn't": "is not", "aren't": "are not", "wasn't": "was not",
    "weren't": "were not", "haven't": "have not", "hasn't": "has not", "hadn't": "had not",
    "couldn't": "could not", "wouldn't": "would not", "shouldn't": "should not",
    "n't": " not", "'re": " are", "'ve": " have", "'ll": " will", "'m": " am"
}

NEGATION_AND_CONTRAST_WORDS = {
    "not", "no", "never", "neither", "nor", "none", "nothing", "nowhere", "hardly",
    "barely", "scarcely", "without", "but", "however", "although", "though", "yet",
    "despite", "unfortunately", "except", "fault"
}

SAFE_STOPWORDS = RAW_NLTK_STOPWORDS - NEGATION_AND_CONTRAST_WORDS

OPINION_LEXICON_POS = {
    "good", "great", "excellent", "amazing", "wonderful", "friendly", "helpful", "clean",
    "spacious", "comfortable", "delicious", "lovely", "perfect", "best", "nice", "fantastic",
    "superb", "polite", "attentive", "convenient", "beautiful", "cozy", "relaxing", "awesome",
    "recommend", "recommended", "happy", "pleased", "impressed", "welcoming", "professional",
    "quiet", "modern", "fresh", "tasty", "worth", "luxury", "luxurious", "outstanding", "warm",
    "praised", "rave", "extensive", "must", "spotless", "exceptional", "courteous", "efficient"
}

OPINION_LEXICON_NEG = {
    "bad", "poor", "terrible", "horrible", "awful", "slow", "rude", "dirty", "noisy", "small",
    "old", "outdated", "broken", "cold", "hard", "uncomfortable", "overpriced", "expensive",
    "worst", "disappointed", "disappointing", "unfriendly", "unhelpful", "smell", "smelly",
    "stain", "stained", "cockroach", "bug", "mold", "issue", "problem", "complaint", "wait",
    "waiting", "delayed", "mediocre", "boring", "crowded", "weak", "fault", "damaged", "filthy"
}

HEDGING_MODIFIERS = {
    "moderate", "okay", "ok", "decent", "fair", "acceptable", "average", "slightly",
    "bit", "ordinary", "normal", "passable", "tolerable", "adequate", "fine", "basic"
}

INTENSIFIERS = {
    "very", "extremely", "really", "super", "highly", "totally", "absolutely", "completely",
    "quite", "pretty", "so", "too", "exceptionally", "incredibly", "amazingly", "remarkably",
    "consistently", "truly", "especially", "particularly"
}

NEGATION_TRIGGERS = {"not", "never", "no", "hardly", "barely"}
NEGATION_SKIP_WORDS = {"very", "too", "so", "that", "really", "at"}

ASPECT_KEYWORDS = {
    "Service": {
        "staff", "service", "services", "member", "members", "reception", "receptionist",
        "front", "desk", "manager", "waiter", "waitress", "concierge", "bellboy", "helpful",
        "friendly", "polite", "check", "checkin", "checkout", "attendant", "housekeeping",
        "host", "crew", "hospitality", "team", "assistance", "welcoming", "professional", "attentive"
    },
    "Facility": {
        "facility", "facilities", "maintenance", "traffic", "location", "building", "property",
        "room", "rooms", "common", "area", "areas", "bed", "beds", "bathroom", "shower",
        "toilet", "air", "ac", "conditioning", "clean", "cleanliness", "noise", "noisy",
        "soundproof", "view", "balcony", "floor", "elevator", "lift", "pillow", "mattress",
        "furniture", "decor", "water", "carpet", "lobby", "parking", "surrounding", "hallway"
    },
    "Experience": {
        "stay", "stayed", "experience", "holiday", "vacation", "trip", "night", "nights",
        "overall", "atmosphere", "vibe", "relax", "relaxing", "enjoy", "enjoyed", "memorable",
        "comfort", "comfortable", "quiet", "peaceful", "feeling", "impression", "journey",
        "moment", "couples", "family", "solo", "honeymoon", "getaway", "place"
    },
    "Amenity": {
        "amenity", "amenities", "kitchen", "food", "foods", "meal", "meals", "dish", "dishes",
        "taste", "tasty", "delicious", "flavor", "dining", "breakfast", "buffet", "pool",
        "swimming", "spa", "gym", "fitness", "wifi", "internet", "restaurant", "bar", "drink",
        "drinks", "coffee", "tea", "lounge", "shuttle", "massage", "rooftop", "menu", "cuisine",
        "activity", "activities"
    },
    "Loyalty": {
        "back", "return", "returned", "returning", "again", "times", "visit", "visits",
        "visited", "revisit", "recommend", "recommended", "definitely", "choose", "future",
        "next", "repeat", "loyal", "regular", "choice", "staying", "book", "booking", "must"
    },
    "Branding": {
        "star", "stars", "luxury", "luxurious", "brand", "chain", "standard", "reputation",
        "class", "five", "four", "management", "value", "price", "pricing", "worth",
        "prestigious", "accor", "sofitel", "pullman", "novotel", "marriott", "hilton",
        "intercontinental", "sheraton", "flagship", "resort"
    }
}

ASPECT_ID_MAP = {asp: idx + 1 for idx, asp in enumerate(cfg.CANONICAL_ASPECTS)}
ASPECT_ID_MAP["None"] = 0
SENTIMENT_TO_ID = {"Negative": 1, "Neutral": 2, "Positive": 3}
ID_TO_SENTIMENT = {0: "Not Mentioned", 1: "Negative", 2: "Neutral", 3: "Positive"}
POLARITY_ID_TO_LABEL = {0: "Negative", 1: "Neutral", 2: "Positive"}

_CLAUSE_SPLIT_RE = re.compile(
    r"(?:[;,]\s*(?P<conj_p>even\s+though|unfortunately|however|although|whereas|though|while|except|but|yet)\b)"
    r"|(?:\s+(?P<conj_w>even\s+though|however|although|whereas|though|while|but|yet)\b)"
    r"|(?P<semi>;+)",
    flags=re.IGNORECASE,
)
_CONTRAST_START_RE = re.compile(
    r"^(?:even\s+though|unfortunately|however|although|whereas|though|while|except|but|yet)\b",
    flags=re.IGNORECASE,
)
_LOYALTY_PHRASE_RE = re.compile(
    r"\b(many times|return visit|came back|come back|been here|stay again|recommend|book early|must go)\b",
    flags=re.IGNORECASE,
)


def nltk_negation_aware_preprocess(text: str) -> Tuple[List[str], List[str]]:
    lower_txt = unicodedata.normalize("NFC", str(text)).lower()
    for k, v in CONTRACTIONS_MAP.items():
        lower_txt = re.sub(r"\b" + re.escape(k) + r"\b", v, lower_txt)
        lower_txt = lower_txt.replace(k, v)
    lower_txt = re.sub(r"@\w+|https?://\S+|www\.\S+", " ", lower_txt)
    raw_tokens = re.findall(r"[a-z]+", lower_txt)
    clean_tokens = [lemmatize_word(t) for t in raw_tokens if t not in SAFE_STOPWORDS and len(t) > 1]
    return raw_tokens, clean_tokens


def preprocess_text_paper1(text: str) -> Tuple[str, str]:
    norm_txt = unicodedata.normalize("NFC", str(text)).lower()
    for k, v in CONTRACTIONS_MAP.items():
        norm_txt = re.sub(r"\b" + re.escape(k) + r"\b", v, norm_txt)
        norm_txt = norm_txt.replace(k, v)
    norm_txt = re.sub(r"@\w+|https?://\S+|www\.\S+", " ", norm_txt)
    clean_review_text = re.sub(r"[^a-z0-9.,!?'\s-]", " ", norm_txt)
    clean_review_text = re.sub(r"\s+", " ", clean_review_text).strip()
    _, clean_tokens = nltk_negation_aware_preprocess(norm_txt)
    paper1_nltk_text = " ".join(clean_tokens)
    return clean_review_text, paper1_nltk_text


def parse_annotations_safe(anns: Any) -> List[Dict[str, Any]]:
    if anns is None:
        return []
    if isinstance(anns, np.ndarray):
        anns = anns.tolist()
    elif isinstance(anns, str):
        try:
            anns = json.loads(anns)
        except Exception:
            return []
    if not isinstance(anns, list):
        return []
    return [dict(a) for a in anns if isinstance(a, dict)]


def _count_negated_lexicon(raw_toks: List[str]) -> Tuple[int, int, int]:
    pos_cnt = sum(1 for t in raw_toks if t in OPINION_LEXICON_POS)
    neg_cnt = sum(1 for t in raw_toks if t in OPINION_LEXICON_NEG)
    hedge_cnt = sum(1 for t in raw_toks if t in HEDGING_MODIFIERS and t not in {"never"})
    for idx_t, tok in enumerate(raw_toks[:-1]):
        if tok in NEGATION_TRIGGERS and raw_toks[idx_t + 1] != "forget":
            nxt = raw_toks[idx_t + 1]
            target_tok = None
            if nxt in OPINION_LEXICON_POS or nxt in OPINION_LEXICON_NEG:
                target_tok = nxt
            elif nxt == "at" and idx_t + 3 < len(raw_toks) and raw_toks[idx_t + 2] == "all" and raw_toks[idx_t + 3] != "forget":
                cand = raw_toks[idx_t + 3]
                if cand in OPINION_LEXICON_POS or cand in OPINION_LEXICON_NEG:
                    target_tok = cand
            elif nxt in NEGATION_SKIP_WORDS and idx_t + 2 < len(raw_toks) and raw_toks[idx_t + 2] != "forget":
                cand = raw_toks[idx_t + 2]
                if cand in OPINION_LEXICON_POS or cand in OPINION_LEXICON_NEG:
                    target_tok = cand
            if target_tok in OPINION_LEXICON_POS:
                pos_cnt = max(0, pos_cnt - 1)
                neg_cnt += 1
            elif target_tok in OPINION_LEXICON_NEG:
                neg_cnt = max(0, neg_cnt - 1)
                pos_cnt += 1
    return pos_cnt, neg_cnt, hedge_cnt


def compute_span_calibrated_sentiment(span_text: str, default_sent_id: int) -> int:
    raw_toks, _ = nltk_negation_aware_preprocess(span_text)
    pos_cnt, neg_cnt, hedge_cnt = _count_negated_lexicon(raw_toks)
    base_sent = int(default_sent_id) if default_sent_id in (1, 2, 3) else 2
    base_sent = max(1, min(3, base_sent))

    if neg_cnt > pos_cnt and neg_cnt >= 1:
        return 1
    if pos_cnt > neg_cnt and pos_cnt >= 1:
        if hedge_cnt > 0 and pos_cnt == 1:
            return 2
        return 3
    if pos_cnt == neg_cnt and pos_cnt > 0:
        return 2
    if hedge_cnt > 0:
        return 2
    return base_sent


def _trim_span_offsets(raw_text: str, s_idx: int, e_idx: int) -> Optional[Tuple[int, int]]:
    while s_idx < e_idx and raw_text[s_idx].isspace():
        s_idx += 1
    while e_idx > s_idx and raw_text[e_idx - 1].isspace():
        e_idx -= 1
    if e_idx <= s_idx:
        return None
    return s_idx, e_idx


def split_into_opinion_clauses(raw_text: str) -> List[Tuple[int, int, str]]:
    text = str(raw_text or "")
    if not text.strip():
        return []

    clauses: List[Tuple[int, int, str]] = []
    for sent_match in re.finditer(r"[^.!?]+[.!?]*", text):
        s_start, s_end = sent_match.start(), sent_match.end()
        trimmed_sent = _trim_span_offsets(text, s_start, s_end)
        if trimmed_sent is None:
            continue
        s_start, s_end = trimmed_sent
        sent_str = text[s_start:s_end]
        if len(sent_str) < 2:
            continue

        raw_segs: List[Tuple[int, int]] = []
        cursor = s_start
        for split_m in _CLAUSE_SPLIT_RE.finditer(sent_str):
            if split_m.group("conj_p") is not None:
                split_pos = split_m.start("conj_p")
                next_cursor = s_start + split_pos
            elif split_m.group("conj_w") is not None:
                split_pos = split_m.start("conj_w")
                conj_low = re.sub(r"\s+", " ", split_m.group("conj_w").lower())
                prefix_words = re.findall(r"[a-z]+", sent_str[:split_pos].lower())
                if conj_low == "though" and prefix_words and prefix_words[-1] == "even":
                    continue
                if conj_low == "while" and prefix_words and prefix_words[-1] == "a":
                    continue
                if conj_low == "yet" and prefix_words and prefix_words[-1] in {"not", "as"}:
                    continue
                next_cursor = s_start + split_pos
            else:
                split_pos = split_m.start("semi")
                next_cursor = s_start + split_m.end("semi")

            abs_m_start = s_start + split_pos
            seg = _trim_span_offsets(text, cursor, abs_m_start)
            if seg is not None:
                seg_s, seg_e = seg
                while seg_e > seg_s and text[seg_e - 1] in {",", ";"}:
                    seg_e -= 1
                seg = _trim_span_offsets(text, seg_s, seg_e)
                if seg is not None:
                    raw_segs.append(seg)
                    cursor = next_cursor
        tail_seg = _trim_span_offsets(text, cursor, s_end)
        if tail_seg is not None:
            seg_s, seg_e = tail_seg
            while seg_e > seg_s and text[seg_e - 1] in {",", ";"}:
                seg_e -= 1
            tail_seg = _trim_span_offsets(text, seg_s, seg_e)
            if tail_seg is not None:
                raw_segs.append(tail_seg)

        if not raw_segs:
            continue

        merged: List[Tuple[int, int]] = []
        for seg_s, seg_e in raw_segs:
            frag = text[seg_s:seg_e]
            n_words = len(re.findall(r"[a-zA-Z]+", frag))
            if merged and n_words < 2:
                prev_s, _ = merged[-1]
                merged[-1] = (prev_s, seg_e)
            else:
                merged.append((seg_s, seg_e))

        if len(merged) >= 2:
            first_s, first_e = merged[0]
            if len(re.findall(r"[a-zA-Z]+", text[first_s:first_e])) < 2:
                _, second_e = merged[1]
                merged[1] = (first_s, second_e)
                merged.pop(0)

        for c_s, c_e in merged:
            trimmed_c = _trim_span_offsets(text, c_s, c_e)
            if trimmed_c is not None:
                cs, ce = trimmed_c
                c_str = text[cs:ce]
                if len(c_str) >= 2:
                    clauses.append((cs, ce, c_str))

    if not clauses and text.strip():
        trimmed_full = _trim_span_offsets(text, 0, len(text))
        if trimmed_full is not None:
            cs, ce = trimmed_full
            clauses.append((cs, ce, text[cs:ce]))
    return clauses


def count_aspect_keyword_hits(aspect_name: str, tokens_set: set, text_str: str) -> int:
    kw_set = ASPECT_KEYWORDS.get(aspect_name, set())
    hits = sum(1 for w in tokens_set if w in kw_set)
    if aspect_name == "Loyalty" and _LOYALTY_PHRASE_RE.search(text_str):
        hits += 2
    return hits


def extract_20_features_for_new_review(review_text: str) -> np.ndarray:
    raw_text = unicodedata.normalize("NFC", str(review_text).strip())
    char_len = max(len(raw_text), 1)
    raw_tokens, clean_tokens = nltk_negation_aware_preprocess(raw_text)
    total_raw_toks = max(len(raw_tokens), 1)

    clean_token_ratio = float(min(len(clean_tokens) / total_raw_toks, 1.0))
    opinion_neg_cnt = sum(
        1 for t in raw_tokens
        if (t in OPINION_LEXICON_POS or t in OPINION_LEXICON_NEG or t in NEGATION_AND_CONTRAST_WORDS or t in INTENSIFIERS)
    )
    opinion_neg_density = float(min(opinion_neg_cnt / total_raw_toks, 1.0))

    tok_counts = Counter(clean_tokens)
    aspect_sims = []
    for asp in cfg.CANONICAL_ASPECTS:
        kw_set = ASPECT_KEYWORDS[asp]
        overlap_w = sum(cnt for tok, cnt in tok_counts.items() if tok in kw_set)
        norm_sim = overlap_w / math.sqrt(len(clean_tokens) * len(kw_set)) if clean_tokens else 0.0
        aspect_sims.append(min(1.0, norm_sim * 3.5))
    top_sims = sorted(aspect_sims, reverse=True)[:3]
    ebr_proto_sim = float(sum(top_sims) / len(top_sims)) if top_sims else 0.0

    sentences = [s.strip() for s in re.split(r"[.!?]+", raw_text) if s.strip()]
    aspect_scores = {}
    matched_sentences_per_aspect = defaultdict(list)
    overlap_flag = 0

    for sent in sentences:
        s_raw, s_clean = nltk_negation_aware_preprocess(sent)
        s_all = set(s_raw) | set(s_clean)
        matched_asps = [asp for asp in cfg.CANONICAL_ASPECTS if any(t in ASPECT_KEYWORDS[asp] for t in s_all)]
        if len(matched_asps) >= 2:
            overlap_flag = 1
        pos_hits, neg_hits, hedge_hits = _count_negated_lexicon(s_raw)
        sent_pol = (pos_hits - neg_hits - 0.25 * hedge_hits) / max(pos_hits + neg_hits + hedge_hits, 1) if (pos_hits + neg_hits + hedge_hits) > 0 else 0.25
        for asp in matched_asps:
            matched_sentences_per_aspect[asp].append((sent, sent_pol))

    for asp in cfg.CANONICAL_ASPECTS:
        items = matched_sentences_per_aspect.get(asp, [])
        aspect_scores[asp] = float(np.clip(np.mean([sc for _, sc in items]), -1.0, 1.0)) if items else 0.0

    active_aspects = [asp for asp in cfg.CANONICAL_ASPECTS if matched_sentences_per_aspect.get(asp)]
    asp_count = len(active_aspects)
    dominant_aspect = ASPECT_ID_MAP.get(active_aspects[0], 0) if active_aspects else 0
    active_vals = [aspect_scores[a] for a in active_aspects] if active_aspects else [0.0]
    net_sentiment = float(np.mean(active_vals))
    sent_conflict = 1 if (any(v > 0.1 for v in active_vals) and any(v < -0.1 for v in active_vals)) else 0
    all_matched_lens = [len(s) for items in matched_sentences_per_aspect.values() for s, _ in items]
    evidence_density = float(min(sum(all_matched_lens) / char_len, 1.0)) if all_matched_lens else clean_token_ratio
    avg_rat_len = float(np.mean(all_matched_lens)) if all_matched_lens else float(char_len / max(asp_count, 1))
    coverage_ratio = float(min(asp_count / 6.0, 1.0))

    neutral_hedge_set = HEDGING_MODIFIERS | {"but", "however", "although", "though", "yet", "despite"}
    neutral_hedging_density = float(min(sum(1 for t in raw_tokens if t in neutral_hedge_set) / total_raw_toks, 1.0))
    consensus_strength = float(abs(net_sentiment))

    pos_cnt, neg_cnt, _ = _count_negated_lexicon(raw_tokens)
    intens_cnt = sum(1 for t in raw_tokens if t in INTENSIFIERS)
    base_pol = (pos_cnt - neg_cnt) / max(pos_cnt + neg_cnt, 1) if (pos_cnt + neg_cnt) > 0 else 0.0
    intens_boost = min(0.35, 0.12 * intens_cnt) * (1.0 if base_pol >= 0 else -1.0) if base_pol != 0 else 0.0
    lexical_polarity_intensity = float(np.clip(base_pol + intens_boost, -1.0, 1.0))

    return np.array([
        asp_count,
        dominant_aspect,
        net_sentiment,
        sent_conflict,
        evidence_density,
        avg_rat_len,
        overlap_flag,
        aspect_scores["Service"],
        aspect_scores["Facility"],
        aspect_scores["Experience"],
        aspect_scores["Amenity"],
        aspect_scores["Loyalty"],
        aspect_scores["Branding"],
        coverage_ratio,
        neutral_hedging_density,
        consensus_strength,
        lexical_polarity_intensity,
        clean_token_ratio,
        opinion_neg_density,
        ebr_proto_sim
    ], dtype=np.float32)


## 3. Nạp Tập dữ liệu Parquet, Khởi tạo Ngân hàng Bằng chứng EBR & Giải phóng RAM


In [ ]:
def derive_review_sentiment_id(anns_raw: Any) -> int:
    anns = parse_annotations_safe(anns_raw)
    pos_c = sum(1 for a in anns if a.get("sentiment") == "Positive")
    neg_c = sum(1 for a in anns if a.get("sentiment") == "Negative")
    if pos_c > neg_c:
        return 2
    if neg_c > pos_c:
        return 0
    return 1

parquet_path = next((Path(p) for p in cfg.PARQUET_CANDIDATES if Path(p).exists()), None)
if parquet_path is None:
    raise FileNotFoundError(f"Không tìm thấy hotel_reviews.parquet tại {cfg.PARQUET_CANDIDATES}!")

df_master = pd.read_parquet(parquet_path)
if "clean_review_text" not in df_master.columns or "paper1_nltk_text" not in df_master.columns:
    cleaned_pairs = [preprocess_text_paper1(txt) for txt in df_master["review_text"]]
    df_master["clean_review_text"] = [p[0] for p in cleaned_pairs]
    df_master["paper1_nltk_text"] = [p[1] for p in cleaned_pairs]
    del cleaned_pairs

missing_feats = [c for c in cfg.FEATURE_COLS_20 if c not in df_master.columns]
if missing_feats:
    computed = np.vstack([
        extract_20_features_for_new_review(txt)
        for txt in df_master["review_text"]
    ])
    for idx_f, col_name in enumerate(cfg.FEATURE_COLS_20):
        if col_name not in df_master.columns:
            df_master[col_name] = computed[:, idx_f]
    del computed
if df_master["feat_dominant_aspect"].dtype == object:
    df_master["feat_dominant_aspect"] = df_master["feat_dominant_aspect"].map(lambda x: ASPECT_ID_MAP.get(str(x), 0)).astype(np.float32)

if "sentiment_id" not in df_master.columns:
    df_master["sentiment_id"] = [derive_review_sentiment_id(a) for a in df_master.get("aspect_annotations", [])]
else:
    df_master["sentiment_id"] = df_master["sentiment_id"].clip(0, 2).astype(int)

df_train = df_master[df_master["split"] == "train"].reset_index(drop=True)
df_val = df_master[df_master["split"] == "val"].reset_index(drop=True)
df_test = df_master[df_master["split"] == "test"].reset_index(drop=True)

feat_mean = df_train[cfg.FEATURE_COLS_20].values.astype(np.float32).mean(axis=0)
feat_std = df_train[cfg.FEATURE_COLS_20].values.astype(np.float32).std(axis=0)
feat_std = np.where(feat_std < 1e-6, 1.0, feat_std)

class EBRExemplarBank:
    def __init__(self, canonical_aspects: List[str]):
        self.aspects = canonical_aspects
        self.bank = {asp: {"Positive": [], "Neutral": [], "Negative": []} for asp in canonical_aspects}
        self.aspect_keyword_weights = {asp: Counter() for asp in canonical_aspects}
        self.global_doc_freq = Counter()
        self.total_spans = 0

    def build_from_dataframe(self, df_tr: pd.DataFrame):
        for _, row in df_tr.iterrows():
            anns = parse_annotations_safe(row.get("aspect_annotations", []))
            for ann in anns:
                asp = ann.get("aspect")
                sent = ann.get("sentiment", "Neutral")
                span_txt = str(ann.get("span_text", "")).strip()
                if asp in self.bank and sent in self.bank[asp] and len(span_txt) >= 8:
                    _, clean_toks = nltk_negation_aware_preprocess(span_txt)
                    if clean_toks:
                        self.total_spans += 1
                        for t in set(clean_toks):
                            self.global_doc_freq[t] += 1
                            self.aspect_keyword_weights[asp][t] += 1
                        self.bank[asp][sent].append({
                            "span_text": span_txt,
                            "tokens": clean_toks
                        })
        for asp in self.aspects:
            for kw in ASPECT_KEYWORDS[asp]:
                self.aspect_keyword_weights[asp][kw] += 25

    def get_top_aspect_terms(self, aspect: str, top_k: int = 35) -> List[str]:
        scores = {}
        total_a = sum(self.aspect_keyword_weights[aspect].values()) + 1.0
        for tok, cnt in self.aspect_keyword_weights[aspect].items():
            tf = cnt / total_a
            idf = math.log((self.total_spans + 1.0) / (self.global_doc_freq.get(tok, 0) + 1.0)) + 1.0
            bonus = 2.0 if tok in ASPECT_KEYWORDS[aspect] else 1.0
            scores[tok] = tf * idf * bonus
        ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        return [w for w, _ in ranked[:top_k]]

ebr_bank = EBRExemplarBank(cfg.CANONICAL_ASPECTS)
ebr_bank.build_from_dataframe(df_train)
with open(cfg.EBR_BANK_SAVE_PATH, "wb") as f:
    pickle.dump(ebr_bank, f)
del df_master
gc.collect()
print(f"Loaded: {parquet_path} | Train: {len(df_train):,} | Val: {len(df_val):,} | Test: {len(df_test):,} | EBR Spans: {ebr_bank.total_spans:,}")


## 4. Thiết kế CRAMABSADataset với Multi-Polarity Targets, Token Polarity Labels & Dynamic Padding


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(cfg.BACKBONE_NAME)

_LEMMA_CACHE: Dict[str, str] = {}
def cached_lemmatize(w: str) -> str:
    res = _LEMMA_CACHE.get(w)
    if res is None:
        res = lemmatize_word(w)
        _LEMMA_CACHE[w] = res
    return res

class CRAMABSADataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer, feat_mean: np.ndarray, feat_std: np.ndarray, max_len: int = 192, max_comment_len: int = 48):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.feat_mean = feat_mean
        self.feat_std = feat_std
        self.max_len = max_len
        self.max_comment_len = max_comment_len
        self.pad_id = self.tokenizer.pad_token_id if self.tokenizer.pad_token_id is not None else 0
        features_raw = self.df[cfg.FEATURE_COLS_20].values.astype(np.float32)
        self.features_norm = (features_raw - self.feat_mean) / self.feat_std
        del features_raw
        self.samples: List[Dict[str, Any]] = []
        self._pretokenize_all()
        self.df = None
        self.features_norm = None
        gc.collect()

    def _pretokenize_all(self):
        n_rows = len(self.df)
        if n_rows == 0:
            return

        raw_texts = [unicodedata.normalize("NFC", str(t)) for t in self.df["review_text"]]
        review_ids = self.df["review_id"].values if "review_id" in self.df.columns else np.arange(n_rows)
        has_sent_col = "sentiment_id" in self.df.columns
        sent_col_vals = self.df["sentiment_id"].values if has_sent_col else None
        anns_col = self.df["aspect_annotations"].tolist() if "aspect_annotations" in self.df.columns else [[] for _ in range(n_rows)]

        batch_enc = self.tokenizer(
            raw_texts,
            max_length=self.max_len,
            padding=False,
            truncation=True,
            return_offsets_mapping=True
        )
        all_input_ids = batch_enc["input_ids"]
        all_attention_masks = batch_enc["attention_mask"]
        all_offsets = batch_enc["offset_mapping"]

        pending_spans: List[Tuple[int, int, str]] = []

        for idx in range(n_rows):
            raw_text = raw_texts[idx]
            feat_vec = self.features_norm[idx]
            ids = all_input_ids[idx]
            mask = all_attention_masks[idx]
            offsets = all_offsets[idx]
            seq_len = len(ids)

            input_ids = torch.tensor(ids, dtype=torch.int32)
            attention_mask = torch.tensor(mask, dtype=torch.uint8)

            if has_sent_col and pd.notna(sent_col_vals[idx]):
                sentiment_id = int(sent_col_vals[idx])
            else:
                sentiment_id = derive_review_sentiment_id(anns_col[idx])
            sentiment_id = max(0, min(cfg.NUM_POLARITY_CLASSES - 1, sentiment_id))

            aspect_sent_labels = torch.zeros(cfg.NUM_ASPECTS, dtype=torch.int8)
            aspect_dual_pol_targets = torch.zeros((cfg.NUM_ASPECTS, 3), dtype=torch.uint8)
            token_polarity_labels = torch.full((seq_len,), -100, dtype=torch.int8)
            rationale_masks = torch.zeros((cfg.NUM_ASPECTS, seq_len), dtype=torch.uint8)
            commentary_target_ids = torch.full(
                (cfg.NUM_ASPECTS, self.max_comment_len),
                self.pad_id,
                dtype=torch.int32
            )

            anns = parse_annotations_safe(anns_col[idx])
            asp_sent_votes = defaultdict(list)
            asp_span_texts = defaultdict(list)
            valid_anns = []

            for ann in anns:
                asp = ann.get("aspect")
                sent = ann.get("sentiment", "Neutral")
                if asp not in ASPECT_ID_MAP or asp == "None":
                    continue
                asp_idx = ASPECT_ID_MAP[asp] - 1
                ann_sent_id = SENTIMENT_TO_ID.get(sent, 2)
                s_char = int(ann.get("start_char", 0))
                e_char = int(ann.get("end_char", 0))
                span_txt = str(ann.get("span_text", "")).strip()

                asp_sent_votes[asp_idx].append(ann_sent_id)
                aspect_dual_pol_targets[asp_idx, ann_sent_id - 1] = 1
                if span_txt:
                    asp_span_texts[asp_idx].append(span_txt)
                    for _, _, sub_str in split_into_opinion_clauses(span_txt):
                        sub_cal = compute_span_calibrated_sentiment(sub_str, ann_sent_id)
                        aspect_dual_pol_targets[asp_idx, sub_cal - 1] = 1
                valid_anns.append((asp, asp_idx, s_char, e_char, ann_sent_id))

            if valid_anns:
                word_spans = []
                for m in re.finditer(r"[a-zA-Z]+", raw_text):
                    w_low = m.group(0).lower()
                    word_spans.append((m.start(), m.end(), w_low, cached_lemmatize(w_low)))

                tok_aspect_kw = {asp: set() for asp in cfg.CANONICAL_ASPECTS}
                for t_i, (tok_s, tok_e) in enumerate(offsets):
                    if tok_e <= tok_s:
                        continue
                    sub_w = re.sub(r"[^a-z]", "", raw_text[tok_s:tok_e].lower())
                    sub_lem = cached_lemmatize(sub_w) if sub_w else ""
                    overlapping_words = [
                        (w_low, w_lem) for (ws, we, w_low, w_lem) in word_spans
                        if tok_s < we and tok_e > ws
                    ]
                    for asp in cfg.CANONICAL_ASPECTS:
                        kw_set = ASPECT_KEYWORDS[asp]
                        if (sub_w and (sub_w in kw_set or sub_lem in kw_set)) or any(
                            (wl in kw_set or wlem in kw_set) for wl, wlem in overlapping_words
                        ):
                            tok_aspect_kw[asp].add(t_i)

                has_multi_asp = len(asp_sent_votes) > 1
                tok_min_dist = np.full((cfg.NUM_ASPECTS, seq_len), 999, dtype=np.int32)
                pending_tok_pol: List[Tuple[int, int, int, bool]] = []

                for asp, asp_idx, s_char, e_char, ann_sent_id in valid_anns:
                    raw_span_slice = raw_text[s_char:e_char]
                    sub_clauses = split_into_opinion_clauses(raw_span_slice)
                    clause_pol_ranges = [
                        (s_char + sub_s, s_char + sub_e, compute_span_calibrated_sentiment(sub_str, ann_sent_id) - 1)
                        for sub_s, sub_e, sub_str in sub_clauses
                    ] if sub_clauses else [(s_char, e_char, ann_sent_id - 1)]
                    span_toks = [
                        t_i for t_i, (tok_s, tok_e) in enumerate(offsets)
                        if tok_e > tok_s and tok_s < e_char and tok_e > s_char
                    ]
                    if not span_toks:
                        continue
                    overlaps_other = has_multi_asp and any(
                        (o_idx != asp_idx and max(s_char, o_s) < min(e_char, o_e))
                        for _, o_idx, o_s, o_e, _ in valid_anns
                    )
                    kw_toks = [t_i for t_i in span_toks if t_i in tok_aspect_kw[asp]]
                    span_char_len = max(1, e_char - s_char)
                    if overlaps_other and kw_toks:
                        for t_i in span_toks:
                            d = min(abs(t_i - k_i) for k_i in kw_toks)
                            if d <= 8:
                                rationale_masks[asp_idx, t_i] = 1
                                if d < tok_min_dist[asp_idx, t_i]:
                                    tok_min_dist[asp_idx, t_i] = d
                                tok_s, tok_e = offsets[t_i]
                                tok_pol = next((cp for cs, ce, cp in clause_pol_ranges if tok_s < ce and tok_e > cs), ann_sent_id - 1)
                                pending_tok_pol.append((t_i, tok_pol, span_char_len, False))
                    else:
                        for t_i in span_toks:
                            rationale_masks[asp_idx, t_i] = 1
                            d = min((abs(t_i - k_i) for k_i in kw_toks), default=50)
                            if d < tok_min_dist[asp_idx, t_i]:
                                tok_min_dist[asp_idx, t_i] = d
                            tok_s, tok_e = offsets[t_i]
                            tok_pol = next((cp for cs, ce, cp in clause_pol_ranges if tok_s < ce and tok_e > cs), ann_sent_id - 1)
                            pending_tok_pol.append((t_i, tok_pol, span_char_len, True))

                # Prioritize exact non-overlapping spans and shorter/more-specific spans
                pending_tok_pol.sort(key=lambda item: (not item[3], item[2]))
                for t_i, pol_cls, _, _ in pending_tok_pol:
                    if token_polarity_labels[t_i] == -100:
                        token_polarity_labels[t_i] = pol_cls

                if has_multi_asp:
                    overlap_tok_indices = torch.where(rationale_masks.sum(dim=0) > 1)[0].tolist()
                    for t_i in overlap_tok_indices:
                        active_asps = [a_i for a_i in range(cfg.NUM_ASPECTS) if rationale_masks[a_i, t_i] > 0]
                        best_a = min(active_asps, key=lambda a_i: tok_min_dist[a_i, t_i])
                        rationale_masks[:, t_i] = 0
                        rationale_masks[best_a, t_i] = 1

            for asp_idx, votes in asp_sent_votes.items():
                maj_sent = Counter(votes).most_common(1)[0][0]
                joined_span = " ".join(asp_span_texts[asp_idx]).strip()
                cal_sent = compute_span_calibrated_sentiment(
                    joined_span if joined_span else raw_text,
                    maj_sent
                )
                aspect_sent_labels[asp_idx] = cal_sent
                if joined_span:
                    pending_spans.append((idx, asp_idx, joined_span))

            corn_targets = torch.tensor(
                [1.0 if sentiment_id > k else 0.0 for k in range(cfg.NUM_POLARITY_CLASSES - 1)],
                dtype=torch.float32
            )

            self.samples.append({
                "review_id": int(review_ids[idx]),
                "input_ids": input_ids,
                "attention_mask": attention_mask,
                "dense_features": torch.tensor(feat_vec, dtype=torch.float32),
                "aspect_sent_labels": aspect_sent_labels,
                "aspect_dual_pol_targets": aspect_dual_pol_targets,
                "token_polarity_labels": token_polarity_labels,
                "rationale_masks": rationale_masks,
                "commentary_target_ids": commentary_target_ids,
                "sentiment_id": torch.tensor(sentiment_id, dtype=torch.int8),
                "corn_targets": corn_targets
            })

        del batch_enc, all_input_ids, all_attention_masks, all_offsets, raw_texts, anns_col

        if pending_spans:
            span_batch_texts = [s_txt for _, _, s_txt in pending_spans]
            span_enc = self.tokenizer(
                span_batch_texts,
                max_length=self.max_comment_len,
                padding="max_length",
                truncation=True,
                return_tensors="pt"
            )["input_ids"].to(torch.int32)
            for p_i, (sample_idx, asp_idx, _) in enumerate(pending_spans):
                self.samples[sample_idx]["commentary_target_ids"][asp_idx] = span_enc[p_i]
            del span_enc, span_batch_texts

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx: int):
        return self.samples[idx]


def make_cram_collate_fn(
    pad_id: int,
    max_seq_len: int,
    pad_to_multiple_of: int = 8,
    dynamic_padding: bool = True
):
    def collate_fn(batch: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        bsz = len(batch)
        max_len_in_batch = max((int(item["input_ids"].numel()) for item in batch), default=1)
        if dynamic_padding:
            target_len = min(
                max_seq_len,
                max(8, math.ceil(max_len_in_batch / pad_to_multiple_of) * pad_to_multiple_of)
            )
        else:
            target_len = max_seq_len

        input_ids = torch.full((bsz, target_len), pad_id, dtype=torch.long)
        attention_mask = torch.zeros((bsz, target_len), dtype=torch.long)
        token_polarity_labels = torch.full((bsz, target_len), -100, dtype=torch.long)
        num_aspects = int(batch[0]["rationale_masks"].size(0))
        rationale_masks = torch.zeros((bsz, num_aspects, target_len), dtype=torch.float32)

        for i, item in enumerate(batch):
            seq_len = min(int(item["input_ids"].numel()), target_len)
            input_ids[i, :seq_len] = item["input_ids"][:seq_len].to(torch.long)
            attention_mask[i, :seq_len] = item["attention_mask"][:seq_len].to(torch.long)
            token_polarity_labels[i, :seq_len] = item["token_polarity_labels"][:seq_len].to(torch.long)
            rationale_masks[i, :, :seq_len] = item["rationale_masks"][:, :seq_len].to(torch.float32)

        return {
            "review_id": torch.tensor([item["review_id"] for item in batch], dtype=torch.long),
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "dense_features": torch.stack([item["dense_features"] for item in batch], dim=0).to(torch.float32),
            "aspect_sent_labels": torch.stack([item["aspect_sent_labels"] for item in batch], dim=0).to(torch.long),
            "aspect_dual_pol_targets": torch.stack([item["aspect_dual_pol_targets"] for item in batch], dim=0).to(torch.float32),
            "token_polarity_labels": token_polarity_labels,
            "rationale_masks": rationale_masks,
            "commentary_target_ids": torch.stack([item["commentary_target_ids"] for item in batch], dim=0).to(torch.long),
            "sentiment_id": torch.stack([item["sentiment_id"] for item in batch], dim=0).to(torch.long),
            "corn_targets": torch.stack([item["corn_targets"] for item in batch], dim=0).to(torch.float32),
        }

    return collate_fn


train_dataset = CRAMABSADataset(df_train, tokenizer, feat_mean, feat_std, cfg.MAX_SEQ_LEN, cfg.MAX_COMMENT_LEN)
val_dataset = CRAMABSADataset(df_val, tokenizer, feat_mean, feat_std, cfg.MAX_SEQ_LEN, cfg.MAX_COMMENT_LEN)
test_dataset = CRAMABSADataset(df_test, tokenizer, feat_mean, feat_std, cfg.MAX_SEQ_LEN, cfg.MAX_COMMENT_LEN)

cram_collate_fn = make_cram_collate_fn(
    pad_id=tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0,
    max_seq_len=cfg.MAX_SEQ_LEN,
    pad_to_multiple_of=8,
    dynamic_padding=getattr(cfg, "USE_DYNAMIC_PADDING", True)
)

train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.BATCH_SIZE,
    shuffle=True,
    num_workers=cfg.NUM_WORKERS,
    pin_memory=cfg.PIN_MEMORY,
    collate_fn=cram_collate_fn
)
val_loader = DataLoader(
    val_dataset,
    batch_size=cfg.EVAL_BATCH_SIZE,
    shuffle=False,
    num_workers=cfg.NUM_WORKERS,
    pin_memory=cfg.PIN_MEMORY,
    collate_fn=cram_collate_fn
)
print(
    f"Train Batches: {len(train_loader)} | Val Batches: {len(val_loader)} | "
    f"eval_bs={cfg.EVAL_BATCH_SIZE} | dynamic_padding={getattr(cfg, 'USE_DYNAMIC_PADDING', True)} | pin_memory={cfg.PIN_MEMORY}"
)


## 5. Kiến trúc CRAM_ABSA_MultiAspectModel (Hybrid Multi-Segment, bfloat16 / SDPA / Gradient Checkpointing & MPS-Safe GRU)


In [ ]:
class CRAM_ABSA_MultiAspectModel(nn.Module):
    def __init__(
        self,
        cfg: TrainConfig,
        attn_implementation: Optional[str] = None,
        enable_grad_ckpt: Optional[bool] = None
    ):
        super().__init__()
        self.num_aspects = cfg.NUM_ASPECTS
        self.num_sent_classes = cfg.NUM_SENTIMENT_CLASSES
        self.num_polarity_classes = cfg.NUM_POLARITY_CLASSES
        self.hidden_dim = cfg.HIDDEN_DIM
        self.max_comment_len = getattr(cfg, "MAX_COMMENT_LEN", 48)
        self.canonical_aspects = list(cfg.CANONICAL_ASPECTS)
        self.has_trained_token_polarity = True
        self.has_trained_multi_pol = True

        attn_impl = (
            attn_implementation
            if attn_implementation is not None
            else ("sdpa" if DEVICE.type == "mps" else "eager")
        )
        try:
            self.bert = AutoModel.from_pretrained(cfg.BACKBONE_NAME, attn_implementation=attn_impl)
        except Exception:
            self.bert = AutoModel.from_pretrained(cfg.BACKBONE_NAME)

        use_gc = getattr(cfg, "USE_GRAD_CHECKPOINTING", False) if enable_grad_ckpt is None else enable_grad_ckpt
        if use_gc and hasattr(self.bert, "gradient_checkpointing_enable"):
            try:
                self.bert.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
            except TypeError:
                self.bert.gradient_checkpointing_enable()

        self.vocab_size = self.bert.config.vocab_size
        self.pad_token_id = self.bert.config.pad_token_id or 0

        self.feature_proj = nn.Sequential(
            nn.Linear(cfg.NUM_FEATURES, cfg.PROJ_DIM),
            nn.LayerNorm(cfg.PROJ_DIM),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(cfg.PROJ_DIM, cfg.HIDDEN_DIM),
            nn.LayerNorm(cfg.HIDDEN_DIM)
        )
        self.residual_gate = nn.Parameter(torch.tensor(0.28))

        self.ebr_aspect_prototypes = nn.Parameter(torch.randn(cfg.NUM_ASPECTS, cfg.HIDDEN_DIM) * 0.02)
        self.aspect_cross_attn = nn.MultiheadAttention(
            embed_dim=cfg.HIDDEN_DIM,
            num_heads=getattr(cfg, "ATTN_HEADS", 8),
            dropout=0.1,
            batch_first=True
        )
        self.rationale_proj = nn.Linear(cfg.HIDDEN_DIM, cfg.HIDDEN_DIM)
        self.rationale_bias = nn.Parameter(torch.zeros(cfg.NUM_ASPECTS, 1))

        self.aspect_classifiers = nn.ModuleList([
            nn.Sequential(
                nn.Linear(cfg.HIDDEN_DIM * 2, cfg.PROJ_DIM),
                nn.LayerNorm(cfg.PROJ_DIM),
                nn.GELU(),
                nn.Dropout(0.20),
                nn.Linear(cfg.PROJ_DIM, cfg.NUM_SENTIMENT_CLASSES)
            )
            for _ in range(cfg.NUM_ASPECTS)
        ])

        self.aspect_multi_pol_head = nn.Sequential(
            nn.Linear(cfg.HIDDEN_DIM * 2, cfg.PROJ_DIM),
            nn.LayerNorm(cfg.PROJ_DIM),
            nn.GELU(),
            nn.Dropout(0.20),
            nn.Linear(cfg.PROJ_DIM, 3)
        )
        self.token_polarity_proj = nn.Linear(cfg.HIDDEN_DIM, 3)

        self.corn_head = nn.Sequential(
            nn.Linear(cfg.HIDDEN_DIM * 2, cfg.PROJ_DIM),
            nn.LayerNorm(cfg.PROJ_DIM),
            nn.GELU(),
            nn.Dropout(0.20),
            nn.Linear(cfg.PROJ_DIM, cfg.NUM_POLARITY_CLASSES - 1)
        )

        self.commentary_gru = nn.GRU(
            input_size=cfg.HIDDEN_DIM,
            hidden_size=cfg.HIDDEN_DIM,
            num_layers=1,
            batch_first=True
        )
        self.commentary_out = nn.Linear(cfg.HIDDEN_DIM, self.vocab_size, bias=False)
        self.commentary_out.weight = self.bert.embeddings.word_embeddings.weight

    @torch.no_grad()
    def initialize_ebr_prototypes(self, bert_tokenizer, ebr_bank_obj: Optional[EBRExemplarBank] = None):
        device = next(self.bert.parameters()).device
        was_training = self.bert.training
        self.bert.eval()
        try:
            raw_vecs = []
            for asp in self.canonical_aspects:
                if ebr_bank_obj is not None:
                    top_terms = ebr_bank_obj.get_top_aspect_terms(asp, top_k=35)
                else:
                    top_terms = sorted(list(ASPECT_KEYWORDS.get(asp, {asp.lower()})))
                proto_str = f"{asp} aspect: " + " ".join(top_terms)
                enc = bert_tokenizer(proto_str, return_tensors="pt", truncation=True, max_length=64).to(device)
                out = self.bert(**enc)
                raw_vecs.append(out.last_hidden_state.mean(dim=1).squeeze(0).float())
            vecs = torch.stack(raw_vecs, dim=0)
            vecs = vecs - vecs.mean(dim=0, keepdim=True)
            vecs = F.normalize(vecs, p=2, dim=-1)
            self.ebr_aspect_prototypes.data.copy_(vecs)
        finally:
            if was_training:
                self.bert.train()

    def forward(self, input_ids, attention_mask, dense_features, commentary_target_ids=None):
        bsz = input_ids.size(0)
        if input_ids.is_cuda:
            self.commentary_gru.flatten_parameters()

        bert_out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        seq_hidden = bert_out.last_hidden_state
        cls_hidden = seq_hidden[:, 0, :]

        # Keep lightweight post-BERT cross-attention & scoring in fp32 on MPS to avoid bf16/baddbmm precision issues
        use_mps_fp32 = (input_ids.device.type == "mps")
        head_ctx = torch.amp.autocast(device_type="mps", enabled=False) if use_mps_fp32 else nullcontext()
        with head_ctx:
            if use_mps_fp32:
                seq_hidden = seq_hidden.float()
                cls_hidden = cls_hidden.float()
                dense_in = dense_features.float()
            else:
                dense_in = dense_features

            feat_emb = self.feature_proj(dense_in)
            conditioned_seq = seq_hidden + self.residual_gate * feat_emb.unsqueeze(1)
            conditioned_cls = cls_hidden + feat_emb

            queries = self.ebr_aspect_prototypes.unsqueeze(0).expand(bsz, -1, -1)
            if queries.dtype != conditioned_seq.dtype:
                queries = queries.to(dtype=conditioned_seq.dtype)
            key_padding_mask = (attention_mask == 0)
            cross_ctx, cross_attn_weights = self.aspect_cross_attn(
                query=queries,
                key=conditioned_seq,
                value=conditioned_seq,
                key_padding_mask=key_padding_mask,
                need_weights=True
            )
            cross_ctx = cross_ctx.float()
            cross_attn_weights = cross_attn_weights.float()

            token_proj = self.rationale_proj(conditioned_seq).float()
            proto_f = self.ebr_aspect_prototypes.float()
            proto_norm = F.normalize(proto_f, p=2, dim=-1)
            token_norm = F.normalize(token_proj, p=2, dim=-1)
            bilinear_scores = torch.einsum("bld,ad->bal", token_proj, proto_f) / math.sqrt(self.hidden_dim)
            cosine_scores = torch.einsum("bld,ad->bal", token_norm, proto_norm)

            valid_len = attention_mask.sum(dim=-1, keepdim=True).unsqueeze(1).float().clamp(min=1.0)
            rel_attn = (cross_attn_weights * valid_len).clamp(min=1e-4)
            raw_rat_logits = bilinear_scores + cosine_scores + 0.5 * torch.log(rel_attn) + self.rationale_bias.float()

            pad_mask = attention_mask.unsqueeze(1).float()
            raw_rat_probs = torch.sigmoid(raw_rat_logits) * pad_mask

            denom = raw_rat_probs.sum(dim=-1, keepdim=True).clamp(min=1e-4)
            rat_pooled = torch.einsum("bal,bld->bad", raw_rat_probs, conditioned_seq.float()) / denom
            masked_pooled = 0.55 * rat_pooled + 0.45 * cross_ctx

            aspect_logits_list = []
            aspect_multi_pol_list = []
            feat_emb_f = feat_emb.float()
            for a_idx in range(self.num_aspects):
                asp_ctx = torch.cat([masked_pooled[:, a_idx, :], feat_emb_f], dim=-1)
                aspect_logits_list.append(self.aspect_classifiers[a_idx](asp_ctx))
                aspect_multi_pol_list.append(self.aspect_multi_pol_head(asp_ctx))

            aspect_logits = torch.stack(aspect_logits_list, dim=1)
            aspect_multi_pol_logits = torch.stack(aspect_multi_pol_list, dim=1)
            token_polarity_logits = self.token_polarity_proj(conditioned_seq.float())

            asp_present_logit = (aspect_logits[:, :, 1:].logsumexp(dim=-1, keepdim=True) - aspect_logits[:, :, 0:1]).detach()
            rationale_logits = raw_rat_logits + 0.35 * asp_present_logit.clamp(min=-6.0, max=4.0)
            rationale_probs = torch.sigmoid(rationale_logits) * pad_mask

            global_aspect_ctx = masked_pooled.mean(dim=1)
            corn_input = torch.cat([conditioned_cls.float(), global_aspect_ctx], dim=-1)
            corn_logits = self.corn_head(corn_input)

        commentary_logits = None
        commentary_loss = torch.zeros(1, device=input_ids.device, dtype=torch.float32)
        if commentary_target_ids is not None:
            _, n_asp, c_len = commentary_target_ids.size()
            flat_targets = commentary_target_ids.reshape(bsz * n_asp, c_len)
            dec_in_ids = flat_targets[:, :-1]
            dec_target_ids = flat_targets[:, 1:]
            non_pad_mask = (dec_target_ids != self.pad_token_id)
            if non_pad_mask.any():
                dec_emb = self.bert.embeddings.word_embeddings(dec_in_ids)
                init_h = masked_pooled.reshape(1, bsz * n_asp, self.hidden_dim).contiguous()
                if dec_emb.device.type == "mps":
                    # Run GRU + tied vocab projection in float32 on MPS to avoid bf16/fp16 autocast precision issues
                    with torch.amp.autocast(device_type="mps", enabled=False):
                        dec_out, _ = self.commentary_gru(dec_emb.float(), init_h.float())
                        active_dec_out = dec_out[non_pad_mask]
                        active_logits = self.commentary_out(active_dec_out.float())
                else:
                    dec_out, _ = self.commentary_gru(dec_emb, init_h)
                    active_dec_out = dec_out[non_pad_mask]
                    active_logits = self.commentary_out(active_dec_out)
                active_targets = dec_target_ids[non_pad_mask]
                commentary_loss = F.cross_entropy(active_logits.float(), active_targets).unsqueeze(0)

        return {
            "aspect_logits": aspect_logits,
            "aspect_multi_pol_logits": aspect_multi_pol_logits,
            "token_polarity_logits": token_polarity_logits,
            "rationale_logits": rationale_logits,
            "rationale_probs": rationale_probs,
            "corn_logits": corn_logits,
            "commentary_logits": commentary_logits,
            "commentary_loss": commentary_loss,
            "aspect_embeddings": masked_pooled
        }

    @torch.no_grad()
    def _decode_pointer_tokens(
        self,
        aspect_idx: int,
        outputs: Dict[str, torch.Tensor],
        input_ids: torch.Tensor,
        valid_len: int,
        tokenizer
    ) -> set:
        tok_ids = input_ids[0, :valid_len]
        h_state = outputs["aspect_embeddings"][:, aspect_idx, :].unsqueeze(0).contiguous()
        cur_tok = torch.tensor([[tokenizer.cls_token_id]], dtype=torch.long, device=input_ids.device)
        gen_ids: List[int] = []

        allowed_tokens = set(tok_ids.tolist()) | {tokenizer.sep_token_id, tokenizer.pad_token_id}
        vocab_mask = torch.full((self.vocab_size,), -1e4, device=input_ids.device, dtype=torch.float32)
        for tid in allowed_tokens:
            vocab_mask[tid] = 0.0

        use_mps_fp32 = (input_ids.device.type == "mps")
        gru_ctx = torch.amp.autocast(device_type="mps", enabled=False) if use_mps_fp32 else nullcontext()
        with gru_ctx:
            if use_mps_fp32:
                h_state = h_state.float()
            for _ in range(self.max_comment_len - 1):
                emb = self.bert.embeddings.word_embeddings(cur_tok)
                if use_mps_fp32:
                    emb = emb.float()
                dec_out, h_state = self.commentary_gru(emb, h_state)
                dec_step = dec_out[:, -1, :].float() if use_mps_fp32 else dec_out[:, -1, :]
                step_logits = self.commentary_out(dec_step).squeeze(0).float() + vocab_mask
                for prev_id in gen_ids[-4:]:
                    step_logits[prev_id] -= 3.0
                next_id = int(torch.argmax(step_logits).item())
                if next_id in (tokenizer.sep_token_id, tokenizer.pad_token_id, tokenizer.cls_token_id):
                    break
                gen_ids.append(next_id)
                cur_tok = torch.tensor([[next_id]], dtype=torch.long, device=input_ids.device)
        return set(gen_ids)

    @torch.no_grad()
    def extract_aspect_segments(
        self,
        aspect_idx: int,
        pred_cls: int,
        outputs: Dict[str, torch.Tensor],
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        tokenizer,
        raw_review_text: Optional[str] = None
    ) -> List[Dict[str, str]]:
        aspect_name = self.canonical_aspects[aspect_idx]
        valid_len = int(attention_mask[0].sum().item())
        tok_ids = input_ids[0, :valid_len]
        scores = outputs["rationale_probs"][0, aspect_idx, :valid_len].detach().float().cpu().numpy()

        tok_pol_probs = None
        if getattr(self, "has_trained_token_polarity", False) and outputs.get("token_polarity_logits") is not None:
            tok_pol_probs = F.softmax(outputs["token_polarity_logits"][0, :valid_len].detach().float(), dim=-1).cpu().numpy()

        gru_tok_set = self._decode_pointer_tokens(aspect_idx, outputs, input_ids, valid_len, tokenizer)

        if raw_review_text is None or not str(raw_review_text).strip():
            raw_review_text = tokenizer.decode(tok_ids[1:max(1, valid_len - 1)], skip_special_tokens=True).strip()

        raw_text = unicodedata.normalize("NFC", str(raw_review_text).strip())
        enc_off = tokenizer(
            raw_text,
            max_length=input_ids.size(1),
            truncation=True,
            return_offsets_mapping=True
        )
        offsets = enc_off["offset_mapping"]

        clauses = split_into_opinion_clauses(raw_text)
        default_pol = pred_cls if pred_cls in (1, 2, 3) else 2
        if not clauses:
            cal_sent = compute_span_calibrated_sentiment(raw_text, default_pol)
            return [{"sentiment": ID_TO_SENTIMENT.get(cal_sent, "Neutral"), "text": raw_text}]

        clause_infos: List[Dict[str, Any]] = []
        for c_idx, (c_start, c_end, c_str) in enumerate(clauses):
            tok_indices = [
                t_i for t_i, (ts, te) in enumerate(offsets)
                if te > ts and ts < c_end and te > c_start and t_i < valid_len
            ]
            rat_sc = float(np.mean([scores[t_i] for t_i in tok_indices])) if tok_indices else 0.0
            gru_ov = sum(1 for t_i in tok_indices if int(tok_ids[t_i].item()) in gru_tok_set) / max(len(tok_indices), 1)
            c_raw_toks, c_clean_toks = nltk_negation_aware_preprocess(c_str)
            c_all_toks = set(c_raw_toks) | set(c_clean_toks)

            kw_hits = count_aspect_keyword_hits(aspect_name, c_all_toks, c_str)
            other_kw_hits = sum(
                count_aspect_keyword_hits(o_asp, c_all_toks, c_str)
                for o_asp in self.canonical_aspects if o_asp != aspect_name
            )
            has_opinion_lex = any((t in OPINION_LEXICON_POS or t in OPINION_LEXICON_NEG) for t in c_raw_toks)
            starts_with_contrast = bool(_CONTRAST_START_RE.match(c_str.strip()))
            ebr_sc = min(1.0, kw_hits * 0.45)
            total_sc = 0.40 * rat_sc + 0.15 * gru_ov + 0.45 * ebr_sc

            clause_infos.append({
                "c_idx": c_idx,
                "text": c_str,
                "tok_indices": tok_indices,
                "rat_sc": rat_sc,
                "gru_ov": gru_ov,
                "kw_hits": kw_hits,
                "other_kw_hits": other_kw_hits,
                "has_opinion_lex": has_opinion_lex,
                "starts_with_contrast": starts_with_contrast,
                "inherited": False,
                "total_sc": total_sc,
            })

        for c_idx in range(1, len(clause_infos)):
            cur = clause_infos[c_idx]
            prev = clause_infos[c_idx - 1]
            prev_relevant = (
                (prev["kw_hits"] > 0 and prev["kw_hits"] >= prev["other_kw_hits"])
                or prev["inherited"]
            )
            if (
                cur["starts_with_contrast"]
                and prev_relevant
                and cur["has_opinion_lex"]
                and cur["kw_hits"] == 0
                and cur["other_kw_hits"] == 0
            ):
                cur["inherited"] = True
                cur["total_sc"] = max(cur["total_sc"], 0.40 * cur["rat_sc"] + 0.15 * cur["gru_ov"] + 0.35)

        explicit_or_inherited = [u for u in clause_infos if (u["kw_hits"] > 0 or u["inherited"])]
        if explicit_or_inherited:
            best_sc = max(u["total_sc"] for u in explicit_or_inherited)
            selected = list(explicit_or_inherited)
            for u in clause_infos:
                if u not in selected and u["other_kw_hits"] == 0 and u["rat_sc"] >= 0.55 and u["total_sc"] >= best_sc * 0.65:
                    selected.append(u)
        else:
            ranked = sorted(clause_infos, key=lambda u: (u["other_kw_hits"] == 0, u["total_sc"]), reverse=True)
            best = ranked[0]
            selected = [best]
            for u in ranked[1:]:
                if u["other_kw_hits"] == 0 and u["rat_sc"] >= max(0.50, best["rat_sc"] * 0.85) and u["total_sc"] >= best["total_sc"] * 0.65:
                    selected.append(u)

        selected.sort(key=lambda u: u["c_idx"])

        grouped_texts: Dict[str, List[str]] = {}
        order_of_labels: List[str] = []
        for u in selected:
            c_str = u["text"]
            cal_sent = compute_span_calibrated_sentiment(c_str, default_pol)
            if tok_pol_probs is not None and u["tok_indices"]:
                cal_1 = compute_span_calibrated_sentiment(c_str, 1)
                cal_3 = compute_span_calibrated_sentiment(c_str, 3)
                if cal_1 != cal_3:
                    mean_tok_pol = tok_pol_probs[u["tok_indices"]].mean(axis=0)
                    prior = np.zeros(3, dtype=np.float32)
                    prior[default_pol - 1] = 1.0
                    blended = 0.35 * prior + 0.65 * mean_tok_pol
                    cal_sent = int(np.argmax(blended)) + 1

            sent_label = ID_TO_SENTIMENT.get(cal_sent, "Neutral")
            if sent_label not in grouped_texts:
                grouped_texts[sent_label] = []
                order_of_labels.append(sent_label)
            grouped_texts[sent_label].append(c_str)

        return [
            {"sentiment": lbl, "text": " ".join(grouped_texts[lbl]).strip()}
            for lbl in order_of_labels
        ]

    @torch.no_grad()
    def generate_aspect_commentary(
        self,
        aspect_idx: int,
        outputs: Dict[str, torch.Tensor],
        input_ids: torch.Tensor,
        attention_mask: torch.Tensor,
        tokenizer,
        raw_review_text: Optional[str] = None,
        pred_cls: int = 2
    ) -> str:
        segments = self.extract_aspect_segments(
            aspect_idx=aspect_idx,
            pred_cls=pred_cls,
            outputs=outputs,
            input_ids=input_ids,
            attention_mask=attention_mask,
            tokenizer=tokenizer,
            raw_review_text=raw_review_text
        )
        joined = " ".join(seg["text"] for seg in segments if seg.get("text")).strip()
        return joined if joined else str(raw_review_text or "").strip()


## 6. Hàm Mất mát Đa nhiệm CRAMABSALoss (v2)


In [ ]:
class CRAMABSALoss(nn.Module):
    def __init__(self, cfg: TrainConfig, class_weights_corn: torch.Tensor, pad_token_id: int = 0):
        super().__init__()
        self.cfg = cfg
        self.register_buffer("corn_weights", class_weights_corn)
        self.aspect_ce = nn.CrossEntropyLoss(label_smoothing=cfg.LABEL_SMOOTHING)
        self.commentary_ce = nn.CrossEntropyLoss(ignore_index=pad_token_id)

    def compute_corn_loss(self, corn_logits: torch.Tensor, corn_targets: torch.Tensor, sentiment_id: torch.Tensor) -> torch.Tensor:
        bce = F.binary_cross_entropy_with_logits(corn_logits.float(), corn_targets.float(), reduction="none")
        sample_w = self.corn_weights[sentiment_id].unsqueeze(-1)
        return (bce * sample_w).mean()

    def compute_overlap_penalty(self, rationale_probs: torch.Tensor) -> torch.Tensor:
        probs_f = rationale_probs.float()
        norm_masks = F.normalize(probs_f, p=2, dim=-1, eps=1e-6)
        gram_matrix = torch.bmm(norm_masks, norm_masks.transpose(1, 2))
        eye = torch.eye(probs_f.size(1), device=probs_f.device).unsqueeze(0)
        off_diag = gram_matrix * (1.0 - eye)
        num_pairs = max(probs_f.size(1) * (probs_f.size(1) - 1), 1)
        gram_pen = off_diag.sum(dim=(1, 2)).mean() / num_pairs
        token_excess = F.relu(probs_f.sum(dim=1) - 1.0).mean()
        return 0.5 * gram_pen + 0.5 * token_excess

    def forward(self, outputs: Dict[str, torch.Tensor], batch: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
        aspect_logits = outputs["aspect_logits"]
        rationale_logits = outputs["rationale_logits"]
        rationale_probs = outputs["rationale_probs"]
        corn_logits = outputs["corn_logits"]
        commentary_logits = outputs.get("commentary_logits")

        aspect_targets = batch["aspect_sent_labels"]
        rationale_targets = batch["rationale_masks"]
        corn_targets = batch["corn_targets"]
        sentiment_id = batch["sentiment_id"]
        pad_mask = batch["attention_mask"].unsqueeze(1).float()

        loss_aspect = self.aspect_ce(aspect_logits.float().view(-1, self.cfg.NUM_SENTIMENT_CLASSES), aspect_targets.view(-1))
        loss_corn = self.compute_corn_loss(corn_logits, corn_targets, sentiment_id)
        loss_overlap = self.compute_overlap_penalty(rationale_probs)

        pos_w = torch.tensor(2.5, device=rationale_logits.device, dtype=torch.float32)
        bce_rat = F.binary_cross_entropy_with_logits(
            rationale_logits.float(), rationale_targets.float(), pos_weight=pos_w, reduction="none"
        )
        valid_aspect_tokens = (pad_mask.sum() * self.cfg.NUM_ASPECTS).clamp(min=1.0)
        loss_rationale = (bce_rat * pad_mask).sum() / valid_aspect_tokens

        loss_commentary = torch.tensor(0.0, device=aspect_logits.device, dtype=torch.float32)
        if outputs.get("commentary_loss") is not None:
            loss_commentary = outputs["commentary_loss"].float().mean()
        elif commentary_logits is not None and "commentary_target_ids" in batch:
            target_seq = batch["commentary_target_ids"][:, :, 1:].contiguous()
            loss_commentary = self.commentary_ce(
                commentary_logits.float().reshape(-1, commentary_logits.size(-1)),
                target_seq.reshape(-1)
            )

        loss_multi_pol = torch.tensor(0.0, device=aspect_logits.device, dtype=torch.float32)
        if "aspect_multi_pol_logits" in outputs and "aspect_dual_pol_targets" in batch:
            mp_pos_w = torch.tensor(4.0, device=aspect_logits.device, dtype=torch.float32)
            loss_multi_pol = F.binary_cross_entropy_with_logits(
                outputs["aspect_multi_pol_logits"].float(),
                batch["aspect_dual_pol_targets"].float(),
                pos_weight=mp_pos_w
            )

        loss_tok_pol = torch.tensor(0.0, device=aspect_logits.device, dtype=torch.float32)
        if "token_polarity_logits" in outputs and "token_polarity_labels" in batch:
            tok_targets = batch["token_polarity_labels"].view(-1)
            if (tok_targets != -100).any():
                loss_tok_pol = F.cross_entropy(
                    outputs["token_polarity_logits"].float().view(-1, 3),
                    tok_targets,
                    ignore_index=-100,
                    label_smoothing=0.02
                )

        total_loss = (
            loss_aspect
            + self.cfg.BETA_CORN * loss_corn
            + self.cfg.BETA_OVERLAP * loss_overlap
            + self.cfg.BETA_RATIONALE * loss_rationale
            + self.cfg.BETA_COMMENTARY * loss_commentary
            + self.cfg.BETA_MULTI_POL * (loss_multi_pol + 0.5 * loss_tok_pol)
        )
        return {
            "total_loss": total_loss,
            "loss_aspect": loss_aspect.detach(),
            "loss_corn": loss_corn.detach(),
            "loss_overlap": loss_overlap.detach(),
            "loss_rationale": loss_rationale.detach(),
            "loss_commentary": loss_commentary.detach(),
            "loss_multi_pol": loss_multi_pol.detach(),
            "loss_tok_pol": loss_tok_pol.detach(),
        }

def decode_corn_predictions(corn_logits: torch.Tensor) -> torch.Tensor:
    probs = torch.sigmoid(corn_logits.float())
    cum_probs = torch.cumprod(probs, dim=-1)
    return (cum_probs >= 0.5).sum(dim=-1)

sent_counts = df_train["sentiment_id"].value_counts().sort_index()
total_tr = len(df_train)
corn_w = torch.tensor(
    [math.sqrt(total_tr / (float(cfg.NUM_POLARITY_CLASSES) * max(sent_counts.get(k, 1), 1))) for k in range(cfg.NUM_POLARITY_CLASSES)],
    dtype=torch.float32,
    device=DEVICE
)
print("CORN 3-class weights:", corn_w.cpu().numpy().round(3))


## 7. Huấn luyện Mô hình CRAM-ABSA v2


In [ ]:
def get_amp_context(enabled: bool):
    if not enabled:
        return nullcontext()
    if DEVICE.type == "mps":
        try:
            return torch.amp.autocast(device_type="mps", dtype=cfg.AMP_DTYPE, enabled=True)
        except Exception:
            return nullcontext()
    if DEVICE.type == "cuda":
        try:
            return torch.amp.autocast(device_type="cuda", dtype=cfg.AMP_DTYPE, enabled=True)
        except Exception:
            return torch.cuda.amp.autocast(enabled=True)
    return nullcontext()


def create_grad_scaler(enabled: bool):
    use_cuda_scaler = bool(enabled and DEVICE.type == "cuda" and cfg.AMP_DTYPE == torch.float16)
    try:
        return torch.amp.GradScaler("cuda", enabled=use_cuda_scaler)
    except Exception:
        return torch.cuda.amp.GradScaler(enabled=use_cuda_scaler)


def get_device_memory_gb() -> float:
    try:
        if DEVICE.type == "mps" and hasattr(torch, "mps") and hasattr(torch.mps, "current_allocated_memory"):
            return float(torch.mps.current_allocated_memory()) / (1024 ** 3)
        if DEVICE.type == "cuda" and torch.cuda.is_available():
            return float(sum(torch.cuda.max_memory_allocated(i) for i in range(NUM_GPUS))) / (1024 ** 3)
    except Exception:
        pass
    return 0.0


def get_mps_driver_gb() -> float:
    try:
        if DEVICE.type == "mps" and hasattr(torch, "mps") and hasattr(torch.mps, "driver_allocated_memory"):
            return float(torch.mps.driver_allocated_memory()) / (1024 ** 3)
    except Exception:
        pass
    return 0.0


def release_accelerator_memory() -> None:
    gc.collect()
    try:
        if DEVICE.type == "mps" and hasattr(torch, "mps") and hasattr(torch.mps, "empty_cache"):
            torch.mps.empty_cache()
        elif DEVICE.type == "cuda" and torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass


def atomic_torch_save(payload: Any, path: Path) -> None:
    target_path = Path(path)
    target_path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = Path(f"{target_path}.tmp")
    try:
        torch.save(payload, tmp_path)
        os.replace(tmp_path, target_path)
    except Exception:
        if tmp_path.exists():
            try:
                tmp_path.unlink()
            except OSError:
                pass
        raise


def atomic_json_save(obj: Any, path: Path) -> None:
    target_path = Path(path)
    target_path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = Path(f"{target_path}.tmp")
    try:
        with open(tmp_path, "w", encoding="utf-8") as f:
            json.dump(obj, f, indent=2, ensure_ascii=False)
            f.flush()
            os.fsync(f.fileno())
        os.replace(tmp_path, target_path)
    except Exception:
        if tmp_path.exists():
            try:
                tmp_path.unlink()
            except OSError:
                pass
        raise


def _move_state_to_cpu(obj: Any) -> Any:
    if isinstance(obj, torch.Tensor):
        return obj.detach().cpu()
    if isinstance(obj, dict):
        return {k: _move_state_to_cpu(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [_move_state_to_cpu(v) for v in obj]
    if isinstance(obj, tuple):
        return tuple(_move_state_to_cpu(v) for v in obj)
    return obj


def _detect_notebook_display():
    try:
        from IPython import get_ipython
        from IPython.display import display
        ip = get_ipython()
        if ip is not None and (
            "IPKernelApp" in getattr(ip, "config", {})
            or "ipykernel" in type(ip).__module__
        ):
            return display
    except Exception:
        pass
    return None


class EpochProgressBar:
    """Single-line 1%-milestone live progress bar for Jupyter IPython and terminal fallback."""

    def __init__(self, epoch: int, total_epochs: int, total_steps: int, width: int = 28):
        self.epoch = int(epoch)
        self.total_epochs = int(total_epochs)
        self.total_steps = max(0, int(total_steps))
        self.width = max(1, int(width))
        self.last_pct = 0
        self.start_time: Optional[float] = None
        self._display_fn = _detect_notebook_display()
        self._handle = None

    @staticmethod
    def _format_duration(seconds: float) -> str:
        if not math.isfinite(seconds) or seconds < 0:
            return "--:--"
        secs = int(round(seconds))
        h, rem = divmod(secs, 3600)
        m, s = divmod(rem, 60)
        if h > 0:
            return f"{h:d}:{m:02d}:{s:02d}"
        return f"{m:02d}:{s:02d}"

    @staticmethod
    def _mem_tag(mem_gb: float) -> str:
        if DEVICE.type == "mps":
            return f"MPS {mem_gb:.2f}GB"
        if DEVICE.type == "cuda":
            return f"VRAM {mem_gb:.2f}GB"
        return f"CPU {mem_gb:.2f}GB"

    def _format_line(self, step: int, pct: int, avg_loss: float, lr: float, mem_gb: float) -> str:
        pct = max(0, min(100, int(pct)))
        filled = min(self.width, (pct * self.width) // 100)
        bar = "█" * filled + "░" * (self.width - filled)
        if step <= 0 or self.start_time is None:
            rate = 0.0
            elapsed_str = "00:00"
            eta_str = "--:--"
        else:
            elapsed = max(time.perf_counter() - self.start_time, 1e-6)
            rate = step / elapsed
            eta = max(0.0, (self.total_steps - step) / rate) if rate > 0 else 0.0
            elapsed_str = self._format_duration(elapsed)
            eta_str = "00:00" if step >= self.total_steps else self._format_duration(eta)
        mem_tag = self._mem_tag(mem_gb)
        return (
            f"[Epoch {self.epoch:02d}/{self.total_epochs:02d}] "
            f"[{bar}] {pct:3d}% | "
            f"step {step}/{self.total_steps} | "
            f"loss={avg_loss:.4f} | "
            f"lr={lr:.2e} | "
            f"{rate:.2f} step/s | "
            f"{elapsed_str}<{eta_str} | "
            f"{mem_tag}"
        )

    def compute_pct(self, step: int) -> int:
        if self.total_steps <= 0 or step >= self.total_steps:
            return 100
        return min(100, int((step * 100) // self.total_steps))

    def needs_update(self, step: int) -> bool:
        pct = self.compute_pct(step)
        return (pct > self.last_pct) or (step >= self.total_steps and self.last_pct < 100)

    def start(self, lr: float, mem_gb: float) -> None:
        self.start_time = time.perf_counter()
        self.last_pct = 0
        line = self._format_line(step=0, pct=0, avg_loss=0.0, lr=lr, mem_gb=mem_gb)
        if self._display_fn is not None:
            try:
                self._handle = self._display_fn({"text/plain": line}, raw=True, display_id=True)
            except Exception:
                self._handle = None
        if self._handle is None:
            sys.stdout.write(f"\r{line}\033[K")
            sys.stdout.flush()

    def update(self, step: int, avg_loss: float, lr: float, mem_gb: float) -> bool:
        pct = self.compute_pct(step)
        if step >= self.total_steps:
            pct = 100
        if pct <= self.last_pct:
            return False
        self.last_pct = pct
        line = self._format_line(step=step, pct=pct, avg_loss=avg_loss, lr=lr, mem_gb=mem_gb)
        if self._handle is not None:
            self._handle.update({"text/plain": line}, raw=True)
        else:
            suffix = "\n" if pct >= 100 else ""
            sys.stdout.write(f"\r{line}\033[K{suffix}")
            sys.stdout.flush()
        return True


NON_BLOCKING = (DEVICE.type == "cuda")

base_model = CRAM_ABSA_MultiAspectModel(cfg).to(DEVICE)
base_model.eval()
base_model.initialize_ebr_prototypes(tokenizer, ebr_bank)
if NUM_GPUS > 1:
    print(f"DataParallel enabled on {NUM_GPUS} GPUs")
    model = nn.DataParallel(base_model, device_ids=list(range(NUM_GPUS)))
else:
    model = base_model

criterion = CRAMABSALoss(cfg, corn_w, pad_token_id=tokenizer.pad_token_id or 0).to(DEVICE)

raw_mod = model.module if isinstance(model, nn.DataParallel) else model
optimizer_grouped_parameters = [
    {"params": raw_mod.bert.parameters(), "lr": cfg.LR_BACKBONE, "weight_decay": cfg.WEIGHT_DECAY},
    {"params": [p for n, p in raw_mod.named_parameters() if not n.startswith("bert.")], "lr": cfg.LR_HEADS, "weight_decay": cfg.WEIGHT_DECAY},
]
optimizer = torch.optim.AdamW(optimizer_grouped_parameters, foreach=False)
total_opt_steps = max(1, math.ceil(len(train_loader) / cfg.GRAD_ACCUM_STEPS) * cfg.EPOCHS)
warmup_steps = int(total_opt_steps * cfg.WARMUP_RATIO)
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_opt_steps,
)
use_cuda_scaler = bool(cfg.USE_AMP and DEVICE.type == "cuda" and cfg.AMP_DTYPE == torch.float16)
scaler = create_grad_scaler(cfg.USE_AMP)


@torch.inference_mode()
def evaluate_cram_absa(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    all_pol_preds, all_pol_trues = [], []
    all_asp_preds, all_asp_trues = [], []
    rat_inter, rat_pred_sum, rat_true_sum = 0.0, 0.0, 0.0
    overlap_rates = []
    mp_preds_list, mp_trues_list = [], []
    tok_pol_correct, tok_pol_total = 0, 0

    eval_device_keys = (
        "input_ids",
        "attention_mask",
        "dense_features",
        "aspect_sent_labels",
        "rationale_masks",
        "corn_targets",
        "sentiment_id",
        "aspect_dual_pol_targets",
        "token_polarity_labels",
    )

    for batch in loader:
        batch_dev = {
            k: batch[k].to(DEVICE, non_blocking=NON_BLOCKING)
            for k in eval_device_keys
            if k in batch and isinstance(batch[k], torch.Tensor)
        }

        with get_amp_context(cfg.USE_AMP):
            outputs = model(
                input_ids=batch_dev["input_ids"],
                attention_mask=batch_dev["attention_mask"],
                dense_features=batch_dev["dense_features"],
                commentary_target_ids=None,
            )
            losses = criterion(outputs, batch_dev)

        eval_loss = (
            F.cross_entropy(
                outputs["aspect_logits"].float().view(-1, cfg.NUM_SENTIMENT_CLASSES),
                batch_dev["aspect_sent_labels"].view(-1),
            ).item()
            + 0.15 * losses["loss_rationale"].item()
        )
        total_loss += eval_loss

        rev_pol_preds = decode_corn_predictions(outputs["corn_logits"]).cpu().numpy()
        rev_pol_trues = batch["sentiment_id"].numpy()
        all_pol_preds.append(rev_pol_preds)
        all_pol_trues.append(rev_pol_trues)

        asp_true_np = batch["aspect_sent_labels"].numpy()
        asp_pol_pred_np = outputs["aspect_logits"][:, :, 1:4].argmax(dim=-1).cpu().numpy()
        active_mask = asp_true_np > 0
        if np.any(active_mask):
            all_pol_trues.append(asp_true_np[active_mask] - 1)
            all_pol_preds.append(asp_pol_pred_np[active_mask])

        asp_preds = outputs["aspect_logits"].argmax(dim=-1)
        all_asp_preds.append(asp_preds.cpu().numpy())
        all_asp_trues.append(asp_true_np)

        active_asp_gate = (asp_preds > 0).unsqueeze(-1).float()
        gated_probs = outputs["rationale_probs"].float() * active_asp_gate
        max_prob_per_tok = gated_probs.max(dim=1, keepdim=True).values
        pred_mask = ((gated_probs >= 0.5) & (gated_probs >= max_prob_per_tok * 0.92)).float()
        true_mask = batch_dev["rationale_masks"]
        rat_inter += (pred_mask * true_mask).sum().item()
        rat_pred_sum += pred_mask.sum().item()
        rat_true_sum += true_mask.sum().item()

        active_asp_per_tok = pred_mask.sum(dim=1)
        leaked_toks = (active_asp_per_tok > 1.0).float().sum().item()
        active_toks = (active_asp_per_tok >= 1.0).float().sum().item()
        overlap_rates.append(leaked_toks / max(active_toks, 1.0))

        if "aspect_multi_pol_logits" in outputs and "aspect_dual_pol_targets" in batch:
            mp_pred = (torch.sigmoid(outputs["aspect_multi_pol_logits"].float()) >= 0.5).cpu().numpy().reshape(-1, 3)
            mp_true = (batch["aspect_dual_pol_targets"].numpy() >= 0.5).reshape(-1, 3)
            mp_preds_list.append(mp_pred)
            mp_trues_list.append(mp_true)

        if "token_polarity_logits" in outputs and "token_polarity_labels" in batch_dev:
            t_lbl = batch_dev["token_polarity_labels"].view(-1)
            t_valid = (t_lbl != -100)
            if t_valid.any():
                t_pred = outputs["token_polarity_logits"].float().view(-1, 3).argmax(dim=-1)
                tok_pol_correct += int((t_pred[t_valid] == t_lbl[t_valid]).sum().item())
                tok_pol_total += int(t_valid.sum().item())

        del outputs, losses, batch_dev

    y_true_pol = np.concatenate(all_pol_trues)
    y_pred_pol = np.concatenate(all_pol_preds)
    acc_at_0 = accuracy_score(y_true_pol, y_pred_pol) * 100.0

    per_cls_mae = [
        np.abs(y_true_pol[y_true_pol == c] - y_pred_pol[y_true_pol == c]).mean()
        for c in range(3) if np.any(y_true_pol == c)
    ]
    macro_mae = float(np.mean(per_cls_mae)) if per_cls_mae else 0.0

    Y_asp_true = np.vstack(all_asp_trues)
    Y_asp_pred = np.vstack(all_asp_preds)
    aspect_f1s = [
        f1_score(Y_asp_true[:, a], Y_asp_pred[:, a], average="macro", zero_division=0) * 100.0
        for a in range(cfg.NUM_ASPECTS)
    ]
    macro_aspect_f1 = float(np.mean(aspect_f1s))
    rationale_f1 = (2.0 * rat_inter / max(rat_pred_sum + rat_true_sum, 1.0)) * 100.0
    leakage_rate = float(np.mean(overlap_rates)) * 100.0

    multi_pol_f1 = 0.0
    if mp_preds_list:
        mp_p = np.vstack(mp_preds_list)
        mp_t = np.vstack(mp_trues_list)
        multi_pol_f1 = f1_score(mp_t, mp_p, average="micro", zero_division=0) * 100.0
    tok_pol_acc = (tok_pol_correct / max(tok_pol_total, 1)) * 100.0

    return {
        "val_loss": total_loss / max(len(loader), 1),
        "acc_at_0": acc_at_0,
        "macro_mae": macro_mae,
        "aspect_macro_f1": macro_aspect_f1,
        "rationale_f1": rationale_f1,
        "leakage_rate": leakage_rate,
        "multi_pol_f1": multi_pol_f1,
        "tok_pol_acc": tok_pol_acc,
        "per_aspect_f1": dict(zip(cfg.CANONICAL_ASPECTS, [round(x, 2) for x in aspect_f1s])),
    }


history = []
best_score = -float("inf")
start_epoch = 1
no_improve_epochs = 0

if getattr(cfg, "RESUME_FROM_LAST", False) and Path(cfg.LAST_CKPT_PATH).exists():
    ckpt = torch.load(cfg.LAST_CKPT_PATH, map_location="cpu", weights_only=False)
    ckpt_backbone = ckpt.get("backbone_name")
    ckpt_seq_len = ckpt.get("max_seq_len")
    if ckpt_backbone != cfg.BACKBONE_NAME or ckpt_seq_len != cfg.MAX_SEQ_LEN:
        del ckpt
        gc.collect()
        raise ValueError(
            f"Resume checkpoint mismatch at {cfg.LAST_CKPT_PATH}: "
            f"expected (backbone={cfg.BACKBONE_NAME}, max_seq_len={cfg.MAX_SEQ_LEN}), "
            f"got (backbone={ckpt_backbone}, max_seq_len={ckpt_seq_len}). "
            f"Delete {Path(cfg.LAST_CKPT_PATH).name} or set RESUME_FROM_LAST = False."
        )
    raw_mod.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    if use_cuda_scaler and "scaler_state_dict" in ckpt:
        scaler.load_state_dict(ckpt["scaler_state_dict"])
    history = list(ckpt.get("history", []))
    best_score = float(ckpt.get("best_score", -float("inf")))
    no_improve_epochs = int(ckpt.get("no_improve_epochs", 0))
    resumed_epoch = int(ckpt["epoch"])
    start_epoch = resumed_epoch + 1
    del ckpt
    gc.collect()
    print(
        f"[Resume] Restored {cfg.LAST_CKPT_PATH} (epoch {resumed_epoch}/{cfg.EPOCHS}) | "
        f"best_score={best_score:.4f} | no_improve={no_improve_epochs} | "
        f"starting at epoch {start_epoch}",
        flush=True,
    )

if start_epoch > cfg.EPOCHS:
    print(
        f"[Resume] Training already completed ({start_epoch - 1}/{cfg.EPOCHS} epochs). "
        f"Best score={best_score:.4f}. Skipping training loop.",
        flush=True,
    )
else:
    for epoch in range(start_epoch, cfg.EPOCHS + 1):
        epoch_t0 = time.perf_counter()
        model.train()
        running_loss_t = torch.zeros((), device=DEVICE, dtype=torch.float32)
        optimizer.zero_grad(set_to_none=True)
        total_steps = len(train_loader)
        if total_steps == 0:
            raise ValueError("train_loader is empty (0 batches).")

        pbar = EpochProgressBar(
            epoch=epoch,
            total_epochs=cfg.EPOCHS,
            total_steps=total_steps,
            width=getattr(cfg, "PROGRESS_BAR_WIDTH", 28),
        )
        pbar.start(
            lr=float(optimizer.param_groups[0]["lr"]),
            mem_gb=get_device_memory_gb(),
        )

        for step, batch in enumerate(train_loader, 1):
            batch_dev = {
                k: (v.to(DEVICE, non_blocking=NON_BLOCKING) if (k != "review_id" and isinstance(v, torch.Tensor)) else v)
                for k, v in batch.items()
            }

            window_start = ((step - 1) // cfg.GRAD_ACCUM_STEPS) * cfg.GRAD_ACCUM_STEPS
            actual_accum = min(cfg.GRAD_ACCUM_STEPS, total_steps - window_start)

            with get_amp_context(cfg.USE_AMP):
                outputs = model(
                    input_ids=batch_dev["input_ids"],
                    attention_mask=batch_dev["attention_mask"],
                    dense_features=batch_dev["dense_features"],
                    commentary_target_ids=batch_dev.get("commentary_target_ids"),
                )
                losses = criterion(outputs, batch_dev)
                loss = losses["total_loss"] / max(actual_accum, 1)

            running_loss_t.add_(losses["total_loss"].detach().float())

            if use_cuda_scaler:
                scaler.scale(loss).backward()
            else:
                loss.backward()

            del outputs, losses, loss, batch_dev

            if step % cfg.GRAD_ACCUM_STEPS == 0 or step == total_steps:
                if use_cuda_scaler:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.MAX_GRAD_NORM)
                    old_scale = scaler.get_scale()
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)
                    if scaler.get_scale() >= old_scale:
                        scheduler.step()
                else:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.MAX_GRAD_NORM)
                    optimizer.step()
                    scheduler.step()
                    optimizer.zero_grad(set_to_none=True)

            if (
                DEVICE.type == "mps"
                and getattr(cfg, "MPS_EMPTY_CACHE_EVERY", 0) > 0
                and step % cfg.MPS_EMPTY_CACHE_EVERY == 0
                and hasattr(torch, "mps")
                and hasattr(torch.mps, "empty_cache")
            ):
                torch.mps.empty_cache()

            if pbar.needs_update(step):
                avg_loss_val = float(running_loss_t.item()) / step
                cur_lr = float(optimizer.param_groups[0]["lr"])
                pbar.update(
                    step=step,
                    avg_loss=avg_loss_val,
                    lr=cur_lr,
                    mem_gb=get_device_memory_gb(),
                )

        train_loss_avg = float(running_loss_t.item()) / max(total_steps, 1)
        val_metrics = evaluate_cram_absa(model, val_loader, criterion)
        mem_used_gb = get_device_memory_gb()
        epoch_time_s = round(time.perf_counter() - epoch_t0, 2)

        composite_score = (
            val_metrics["aspect_macro_f1"]
            + val_metrics["acc_at_0"]
            - 20.0 * val_metrics["macro_mae"]
        )
        is_best = composite_score > best_score
        if is_best:
            best_score = composite_score
            no_improve_epochs = 0
            raw_state_cpu = {k: v.detach().cpu() for k, v in raw_mod.state_dict().items()}
            ckpt_payload = {
                "epoch": epoch,
                "model_name": f"CRAM-ABSA ({cfg.MODEL_FILENAME})",
                "architecture": "cram_absa",
                "max_seq_len": cfg.MAX_SEQ_LEN,
                "model_state_dict": raw_state_cpu,
                "feat_mean": feat_mean,
                "feat_std": feat_std,
                "feature_cols_20": cfg.FEATURE_COLS_20,
                "canonical_aspects": cfg.CANONICAL_ASPECTS,
                "val_metrics": val_metrics,
            }
            atomic_torch_save(ckpt_payload, cfg.MODEL_SAVE_PATH)
            del raw_state_cpu, ckpt_payload
            gc.collect()
        else:
            no_improve_epochs += 1

        epoch_record = {
            "epoch": epoch,
            "train_loss": round(train_loss_avg, 4),
            **{k: (round(v, 4) if isinstance(v, float) else v) for k, v in val_metrics.items()},
            "vram_gb": round(mem_used_gb, 2),
            "epoch_time_s": epoch_time_s,
        }
        history.append(epoch_record)
        atomic_json_save(history, cfg.HISTORY_SAVE_PATH)

        last_state_cpu = {k: v.detach().cpu() for k, v in raw_mod.state_dict().items()}
        last_ckpt_payload = {
            "epoch": epoch,
            "backbone_name": cfg.BACKBONE_NAME,
            "max_seq_len": cfg.MAX_SEQ_LEN,
            "model_state_dict": last_state_cpu,
            "optimizer_state_dict": _move_state_to_cpu(optimizer.state_dict()),
            "scheduler_state_dict": scheduler.state_dict(),
            "history": history,
            "best_score": best_score,
            "no_improve_epochs": no_improve_epochs,
        }
        if use_cuda_scaler:
            last_ckpt_payload["scaler_state_dict"] = scaler.state_dict()
        atomic_torch_save(last_ckpt_payload, cfg.LAST_CKPT_PATH)
        del last_state_cpu, last_ckpt_payload
        gc.collect()

        patience = int(getattr(cfg, "EARLY_STOPPING_PATIENCE", 0) or 0)
        if is_best:
            best_tag = " [BEST]"
        elif patience > 0:
            best_tag = f" [no-improve {no_improve_epochs}/{patience}]"
        else:
            best_tag = ""
        mem_label = "MPS" if DEVICE.type == "mps" else "Mem"
        print(
            f"[Epoch {epoch:02d}/{cfg.EPOCHS:02d}] Summary ({epoch_time_s:.1f}s) | Train Loss: {epoch_record['train_loss']:.4f} | "
            f"Val Loss: {val_metrics['val_loss']:.4f} | Acc@0: {val_metrics['acc_at_0']:.2f}% | "
            f"6-Aspect F1: {val_metrics['aspect_macro_f1']:.2f}% | Macro-MAE: {val_metrics['macro_mae']:.4f} | "
            f"Rat-F1: {val_metrics['rationale_f1']:.2f}% | Leakage: {val_metrics['leakage_rate']:.2f}% | "
            f"MP-F1: {val_metrics['multi_pol_f1']:.2f}% | TokPol: {val_metrics['tok_pol_acc']:.2f}% | "
            f"{mem_label}: {mem_used_gb:.2f} GB{best_tag}",
            flush=True,
        )
        release_accelerator_memory()

        if patience > 0 and no_improve_epochs >= patience:
            print(
                f"[Early Stopping] Triggered at epoch {epoch:02d}/{cfg.EPOCHS:02d} "
                f"({no_improve_epochs} consecutive epochs without improvement; best_score={best_score:.4f}).",
                flush=True,
            )
            break

if history:
    atomic_json_save(history, cfg.HISTORY_SAVE_PATH)
print(f"Saved best model to: {cfg.MODEL_SAVE_PATH}")


## 8. Suy luận Review Mới với Phân tách Praise/Complaint Đa Đoạn theo Từng Khía cạnh


In [ ]:
@torch.no_grad()
def generate_structured_review_output(
    new_review_text: str,
    model: nn.Module,
    tokenizer,
    feat_mean: Optional[np.ndarray] = None,
    feat_std: Optional[np.ndarray] = None,
    max_length: Optional[int] = None
) -> Dict[str, Any]:
    raw_model = model.module if isinstance(model, nn.DataParallel) else model
    raw_model.eval()

    seq_max_len = int(max_length) if max_length is not None else cfg.MAX_SEQ_LEN
    raw_text = unicodedata.normalize("NFC", str(new_review_text).strip())
    raw_feat_20 = extract_20_features_for_new_review(raw_text)
    if feat_mean is not None and feat_std is not None:
        norm_feat_20 = (raw_feat_20 - feat_mean) / np.where(feat_std < 1e-6, 1.0, feat_std)
    else:
        norm_feat_20 = raw_feat_20

    enc = tokenizer(
        raw_text,
        max_length=seq_max_len,
        padding=False,
        truncation=True,
        return_tensors="pt"
    )
    input_ids = enc["input_ids"].to(DEVICE)
    attention_mask = enc["attention_mask"].to(DEVICE)
    dense_tensor = torch.tensor(norm_feat_20, dtype=torch.float32, device=DEVICE).unsqueeze(0)

    outputs = raw_model(input_ids=input_ids, attention_mask=attention_mask, dense_features=dense_tensor)
    pred_global_pol = int(decode_corn_predictions(outputs["corn_logits"]).item())
    overall_sentiment = POLARITY_ID_TO_LABEL.get(pred_global_pol, "Neutral")
    asp_probs = F.softmax(outputs["aspect_logits"].squeeze(0).float(), dim=-1).cpu().numpy()

    multi_pol_probs = None
    if getattr(raw_model, "has_trained_multi_pol", False) and outputs.get("aspect_multi_pol_logits") is not None:
        multi_pol_probs = torch.sigmoid(outputs["aspect_multi_pol_logits"].squeeze(0).float()).cpu().numpy()

    mentioned_aspects = []
    aspect_outputs = {}

    for a_idx, asp in enumerate(cfg.CANONICAL_ASPECTS):
        probs_4 = asp_probs[a_idx]
        pred_cls = int(np.argmax(probs_4))
        multi_active = bool(multi_pol_probs is not None and multi_pol_probs[a_idx].max() >= 0.65 and probs_4[0] < 0.65)
        is_mentioned = (pred_cls != 0) or multi_active

        if is_mentioned:
            mentioned_aspects.append(asp)
            fallback_cls = pred_cls if pred_cls != 0 else (int(np.argmax(multi_pol_probs[a_idx])) + 1)
            segments = raw_model.extract_aspect_segments(
                aspect_idx=a_idx,
                pred_cls=fallback_cls,
                outputs=outputs,
                input_ids=input_ids,
                attention_mask=attention_mask,
                tokenizer=tokenizer,
                raw_review_text=raw_text
            )
            if not segments:
                sent_label = ID_TO_SENTIMENT[fallback_cls]
                segments = [{"sentiment": sent_label, "text": raw_text}]
            aspect_outputs[asp] = {
                "mentioned": True,
                "sentiment": segments[0]["sentiment"] if len(segments) == 1 else "Mixed",
                "segments": segments,
                "text": " ".join(f"{s['sentiment']} - {s['text']}" for s in segments)
            }
        else:
            aspect_outputs[asp] = {
                "mentioned": False,
                "sentiment": "Not Mentioned",
                "segments": [],
                "text": ""
            }

    aspect_list_str = ", ".join(mentioned_aspects) if mentioned_aspects else "None"
    lines = [f"- Aspect: {aspect_list_str}"]
    for asp in cfg.CANONICAL_ASPECTS:
        info = aspect_outputs[asp]
        segs = info["segments"]
        if info["mentioned"] and len(segs) >= 2:
            lines.append(f"- {asp}:")
            for seg in segs:
                lines.append(f"  + {seg['sentiment']} - {seg['text']}")
        elif info["mentioned"] and len(segs) == 1:
            lines.append(f"- {asp}: {segs[0]['sentiment']} - {segs[0]['text']}")
        else:
            lines.append(f"- {asp}:")

    return {
        "Aspect": mentioned_aspects,
        "overall_sentiment": overall_sentiment,
        "aspects": aspect_outputs,
        "formatted_text": "\n".join(lines)
    }


def load_trained_model_and_predict(new_review_text: str, checkpoint_path: Optional[str] = None) -> str:
    ckpt_file = Path(checkpoint_path) if checkpoint_path else cfg.MODEL_SAVE_PATH
    if not ckpt_file.exists():
        raise FileNotFoundError(f"Model checkpoint not found: {ckpt_file}")

    ckpt = torch.load(ckpt_file, map_location="cpu", weights_only=False)
    infer_model = CRAM_ABSA_MultiAspectModel(cfg, enable_grad_ckpt=False)
    raw_sd = ckpt["model_state_dict"]
    infer_model.load_state_dict(raw_sd, strict=False)
    infer_model.has_trained_token_polarity = ("token_polarity_proj.weight" in raw_sd)
    infer_model.has_trained_multi_pol = (
        "aspect_multi_pol_head.weight" in raw_sd
        or "aspect_multi_pol_head.0.weight" in raw_sd
    )
    infer_model = infer_model.to(DEVICE)
    infer_model.eval()

    raw_max_len = ckpt.get("max_seq_len", cfg.MAX_SEQ_LEN)
    max_length = int(raw_max_len) if raw_max_len is not None else cfg.MAX_SEQ_LEN
    f_mean = ckpt.get("feat_mean", globals().get("feat_mean", None))
    f_std = ckpt.get("feat_std", globals().get("feat_std", None))
    del ckpt, raw_sd

    tok = globals().get("tokenizer", None)
    if tok is None:
        tok = AutoTokenizer.from_pretrained(cfg.BACKBONE_NAME)

    res = generate_structured_review_output(
        new_review_text=new_review_text,
        model=infer_model,
        tokenizer=tok,
        feat_mean=f_mean,
        feat_std=f_std,
        max_length=max_length
    )
    return res["formatted_text"]


sample_review = (
    "The front desk staff were extremely welcoming and check-in was fast, "
    "though the dinner waiter was slow and rude. "
    "Our room was spacious and spotless, but the air conditioning was broken and noisy at night. "
    "Breakfast buffet had delicious local dishes and great coffee. "
    "We will definitely return on our next trip!"
)
if cfg.MODEL_SAVE_PATH.exists():
    print(load_trained_model_and_predict(sample_review))
elif "model" in globals():
    demo_res = generate_structured_review_output(sample_review, model, tokenizer, feat_mean, feat_std)
    print(demo_res["formatted_text"])
else:
    print("Skipping sample inference: no saved model checkpoint or in-memory model found.")


## 9. Xuất Mô hình CoreML (.mlpackage) cho Apple Neural Engine (ANE) (Tuỳ chọn)


In [ ]:
class CRAM_ABSA_ANE_Wrapper(nn.Module):
    def __init__(self, base_cram_model: CRAM_ABSA_MultiAspectModel):
        super().__init__()
        self.model = base_cram_model

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor, dense_features: torch.Tensor):
        out = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            dense_features=dense_features
        )
        return (
            out["aspect_logits"],
            out["rationale_probs"],
            out["corn_logits"],
            out["aspect_multi_pol_logits"],
        )


try:
    import coremltools as ct

    export_model = CRAM_ABSA_MultiAspectModel(
        cfg,
        attn_implementation="eager",
        enable_grad_ckpt=False
    ).cpu().eval()
    if cfg.MODEL_SAVE_PATH.exists():
        ckpt_ane = torch.load(cfg.MODEL_SAVE_PATH, map_location="cpu", weights_only=False)
        export_model.load_state_dict(ckpt_ane["model_state_dict"], strict=False)
        raw_ane_len = ckpt_ane.get("max_seq_len", cfg.MAX_SEQ_LEN)
        ane_seq_len = int(raw_ane_len) if raw_ane_len is not None else cfg.MAX_SEQ_LEN
        ane_feat_mean = ckpt_ane.get("feat_mean", globals().get("feat_mean", None))
        ane_feat_std = ckpt_ane.get("feat_std", globals().get("feat_std", None))
        del ckpt_ane
    else:
        src_mod = model.module if isinstance(model, nn.DataParallel) else model
        export_model.load_state_dict({k: v.detach().cpu() for k, v in src_mod.state_dict().items()}, strict=False)
        ane_seq_len = cfg.MAX_SEQ_LEN
        ane_feat_mean = globals().get("feat_mean", None)
        ane_feat_std = globals().get("feat_std", None)

    ane_wrapper = CRAM_ABSA_ANE_Wrapper(export_model).eval()
    dummy_ids = torch.zeros((1, ane_seq_len), dtype=torch.int32)
    dummy_mask = torch.ones((1, ane_seq_len), dtype=torch.int32)
    dummy_feats = torch.zeros((1, cfg.NUM_FEATURES), dtype=torch.float32)

    with torch.no_grad():
        traced_ane = torch.jit.trace(ane_wrapper, (dummy_ids, dummy_mask, dummy_feats), strict=False)

    mlmodel = ct.convert(
        traced_ane,
        inputs=[
            ct.TensorType(name="input_ids", shape=(1, ane_seq_len), dtype=np.int32),
            ct.TensorType(name="attention_mask", shape=(1, ane_seq_len), dtype=np.int32),
            ct.TensorType(name="dense_features", shape=(1, cfg.NUM_FEATURES), dtype=np.float32),
        ],
        outputs=[
            ct.TensorType(name="aspect_logits"),
            ct.TensorType(name="rationale_probs"),
            ct.TensorType(name="corn_logits"),
            ct.TensorType(name="aspect_multi_pol_logits"),
        ],
        convert_to="mlprogram",
        compute_units=ct.ComputeUnit.ALL,
        minimum_deployment_target=ct.target.macOS14,
    )
    mlmodel.save(str(cfg.ANE_SAVE_PATH))
    print(f"Exported CoreML ANE model to: {cfg.ANE_SAVE_PATH}")

    # Brief ANE inference verification & latency check
    loaded_mlmodel = ct.models.MLModel(str(cfg.ANE_SAVE_PATH), compute_units=ct.ComputeUnit.ALL)
    ane_tok = globals().get("tokenizer", None) or AutoTokenizer.from_pretrained(cfg.BACKBONE_NAME)
    enc_test = ane_tok(
        sample_review,
        max_length=ane_seq_len,
        padding="max_length",
        truncation=True,
        return_tensors="np"
    )
    raw_f20 = extract_20_features_for_new_review(sample_review)
    if ane_feat_mean is not None and ane_feat_std is not None:
        norm_f20 = ((raw_f20 - ane_feat_mean) / np.where(ane_feat_std < 1e-6, 1.0, ane_feat_std)).astype(np.float32)[None, :]
    else:
        norm_f20 = raw_f20.astype(np.float32)[None, :]
    ane_inputs = {
        "input_ids": enc_test["input_ids"].astype(np.int32),
        "attention_mask": enc_test["attention_mask"].astype(np.int32),
        "dense_features": norm_f20,
    }
    _ = loaded_mlmodel.predict(ane_inputs)  # warmup
    t0 = time.perf_counter()
    for _ in range(10):
        ane_out = loaded_mlmodel.predict(ane_inputs)
    dt_ms = (time.perf_counter() - t0) * 100.0
    print(
        f"ANE inference check OK | Latency: {dt_ms:.2f} ms/review | "
        f"aspect_logits={ane_out['aspect_logits'].shape}, multi_pol={ane_out['aspect_multi_pol_logits'].shape}"
    )
except ImportError:
    print("coremltools is not installed; skipping optional CoreML ANE export (.mlpackage). Install with: uv pip install coremltools")
